# Unknown-format CRF digitizer - code-generation induction

For every CRF PDF in the input managed folder - **any vendor format, no template,
no configuration** - a pure-Python pass clusters the pages by structural layout
(word-blind typography tokens, per-document chrome damping, similarity threshold
self-selected per document) and picks a handful of representative pages (~4-12);
the LLM (via LLM Mesh) writes a document-specific Python parser from those pages;
the parser runs in a sandboxed subprocess over the whole document; machine gates
plus an LLM page-grounded audit challenge the result; and the LLM revises its own
code in a bounded loop. Output per document: `(form_name, field_name, page)`
records. OIDs are resolved downstream by name mapping against the rule library.

**Requirements**
- Code env for this notebook (the sandbox subprocess uses the same interpreter):
  `PyMuPDF`, `pandas`; `rapidfuzz` optional (mapping cell only).
- Managed folders: one with the input CRF PDFs, one (empty) for output artifacts.
- LLM Mesh: uses the project variable `default_llm_model` (Claude Sonnet 4.5)
  unless `LLM_ID` overrides it below.

**LLM budget - bounded, never endless.** Per document: at most `MAX_VERSIONS`
parser versions (1 generation call each, +1 audit call when gates pass, +1 audit
reprompt if a reply is malformed/partial) plus at most one coverage-confirmation
call. Defaults give a worst case of ~16 calls per document; the benchmark runs
took 7-10. The loop stops early on a clean audit (converged) or when a version
fails to improve on the previous one (diminishing returns; two consecutive
gate-failed versions keep revising until the cap); the best-scoring version -
never merely the last - is exported.

*This notebook is generated by `experiments/recipe_prototype/build_dataiku_notebook.py`.
The "pipeline module" cells below are verbatim copies of the tested repo sources -
edit the repo and regenerate instead of editing them here.*


In [ ]:
# ------------------------------- CONFIG -------------------------------------
INPUT_FOLDER = 'crf_unknown_pdfs'    # managed folder (name or id) with input CRF PDFs
OUTPUT_FOLDER = 'ecs_codegen_out'    # managed folder (name or id) for all artifacts
LLM_ID = None                        # None -> project variable 'default_llm_model'
COMPLETION_SETTINGS = {'temperature': 0.2, 'maxOutputTokens': 8000}  # best effort

MAX_VERSIONS = 5     # hard cap on parser versions per document
DOC_FILTER = ''      # substring filter on PDF names ('' = all)
MAX_DOCS = None      # int -> cap the number of documents (smoke runs)

UPLOAD_PAGE_PNGS = False   # representative-page PNGs are handy but heavy
RUN_OID_MAPPING = False    # name->OID funnel + LLM ranker (see mapping cell)
ECS_INDEX_DATASET = 'ecs_index_data'  # dataset with form_field_value / variable_name

WORK_DIR = None      # None -> ./crf_codegen_work under the kernel's cwd


In [ ]:
# Bootstrap: materialize the embedded pipeline modules into a local work dir,
# so this notebook is fully standalone (no repo checkout on the DSS host).
import os
import sys

WORK = os.path.abspath(WORK_DIR or 'crf_codegen_work')
MODULES_DIR = os.path.join(WORK, 'modules')
os.makedirs(MODULES_DIR, exist_ok=True)
# pipeline paths (input staging dir, output dir) derive from this env var;
# it must be set BEFORE the modules are imported
os.environ['ECS_BASE'] = WORK

def _write_module(name, source):
    with open(os.path.join(MODULES_DIR, name), 'w', encoding='utf-8') as f:
        f.write(source)
    print('module written:', name, '(' + str(len(source)) + ' chars)')


In [ ]:
# --- pipeline module: common.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('common.py', r'''"""Shared page model (Line / build_page_lines / dump_rep_page) plus the V1
five-signal layout fingerprint, kept as a reference baseline.

Pipeline stages:
  stage 0  - cluster pages by structural layout, pick representative pages.
             SHIPPED front-end: generic_profile.cluster_pages_generic (word-blind
             typography tokens, per-document chrome damping, weighted-Jaccard
             leader clustering, per-document theta by stability selection).
             page_fingerprint/cluster_pages below are the v1 five-signal method,
             retained for comparison probes (generic_cluster_probe, qsc_merge_diag).
  stage 1  - LLM writes a document-specific extraction program from the
             representative pages (codegen.py) and revises it in a bounded loop
  stage 2  - the accepted program replays deterministically over every page
"""
from __future__ import annotations

import hashlib
import os
import re
from collections import defaultdict
from dataclasses import dataclass

import fitz

# repo root = two levels above this file; ECS_BASE env var overrides (the Dataiku
# notebook does not use these paths at all - it reads from managed folders)
BASE = os.environ.get("ECS_BASE") or os.path.dirname(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
CRF_DIR = os.path.join(BASE, "data", "crf_forms")
OUT_DIR = os.path.join(BASE, "experiments", "recipe_prototype", "out")


@dataclass
class Line:
    text: str
    x0: float
    y0: float
    x1: float
    y1: float
    size: float
    colors: tuple
    bold: bool

    @property
    def non_black(self) -> bool:
        return any(c != 0 for c in self.colors)


def build_page_lines(page) -> list[Line]:
    """Visual lines with geometry, font size, colour and boldness."""
    d = page.get_text("dict")
    lines: list[Line] = []
    for block in d.get("blocks", []):
        if block.get("type") != 0:
            continue
        for l in block.get("lines", []):
            spans = l.get("spans") or []
            text = "".join(s.get("text", "") for s in spans).strip()
            if not text:
                continue
            x0, y0, x1, y1 = l["bbox"]
            size = max(round(s.get("size", 0.0), 1) for s in spans)
            colors = tuple(sorted({s.get("color", 0) for s in spans}))
            # name substring OR the font-descriptor bold bit (flags 2**4): many
            # non-Latin/embedded fonts carry weight in the descriptor but not in
            # the name (e.g. CJK "...-W6"), which would blind .bold entirely
            bold = any("bold" in (s.get("font", "") or "").lower()
                       or (s.get("flags", 0) & 16) for s in spans)
            lines.append(Line(text, x0, y0, x1, y1, size, colors, bold))
    lines.sort(key=lambda L: (round(L.y0, 1), L.x0))
    return lines


def group_rows(lines: list[Line], ytol: float = 3.5) -> list[list[Line]]:
    """Group lines that sit on the same visual row (same y within tolerance)."""
    rows: list[list[Line]] = []
    anchor = None
    for L in lines:
        if anchor is None or L.y0 - anchor > ytol:
            rows.append([L])
            anchor = L.y0
        else:
            rows[-1].append(L)
    for r in rows:
        r.sort(key=lambda L: L.x0)
    return rows


# ---- V1 five-signal fingerprint (reference baseline; superseded in the shipped
# pipeline by generic_profile.py). HEURISTIC bucketing features, not extraction
# logic: a wrong bucket only means a layout gets split into two clusters
# (costing one extra representative page), never a wrong extraction.
# BRACKET_LINE ([...]-only lines) is a common annotation convention but not
# universal - it is one signal among five, not a requirement.
BRACKET_LINE = re.compile(r"^\[[^\]]+\]")
INT_ONLY = re.compile(r"^\d{1,3}(\.\d)?$")


def _bucket(x: float) -> str:
    return "0" if x == 0 else "lo" if x < 0.25 else "hi"


def page_fingerprint(lines: list[Line], page_width: float) -> tuple:
    """Coarse layout signature built from structure only (never from page content),
    so that e.g. 900 form pages with different questions land in one cluster.

    All cutoffs below (0.25 bucket split, 10% column-presence, 10/40/90 density
    bands, 4 x-bins) are tuned coarseness knobs, not correctness constraints:
    they trade cluster count against representative-page count. Documents with
    unusual line densities may over/under-merge; the coverage-confirm and audit
    rounds are the safety net for that, not these numbers."""
    if not lines:
        return ("<empty>",)
    n = len(lines)
    bracket = _bucket(sum(1 for L in lines if BRACKET_LINE.match(L.text)) / n)
    ints = _bucket(sum(1 for L in lines if INT_ONLY.match(L.text)) / n)
    color = _bucket(sum(1 for L in lines if L.non_black) / n)
    nbins = 4
    xhist = [0] * nbins
    for L in lines:
        xhist[min(nbins - 1, int(L.x0 / max(page_width, 1) * nbins))] += 1
    xsig = "".join("1" if c >= max(2, n * 0.10) else "0" for c in xhist)
    size_bucket = "s" if n <= 10 else "m" if n <= 40 else "l" if n <= 90 else "xl"
    return (bracket, ints, color, size_bucket, xsig)


def cluster_pages(doc, max_reps: int = 10, coverage: float = 0.95) -> dict:
    """V1 exact-tuple clustering (reference baseline - the shipped pipeline calls
    generic_profile.cluster_pages_generic instead). Assign every page to a layout
    cluster; pick representatives from the biggest clusters until `coverage` of
    pages is represented (capped at `max_reps`)."""
    sigs: dict[tuple, list[int]] = defaultdict(list)
    page_lines: dict[int, list[Line]] = {}
    for i in range(doc.page_count):
        page = doc[i]
        lines = build_page_lines(page)
        page_lines[i] = lines
        sigs[page_fingerprint(lines, page.rect.width)].append(i)

    ordered = sorted(sigs.items(), key=lambda kv: -len(kv[1]))
    clusters, covered, reps = [], 0, []
    for sig, pages in ordered:
        is_rep_cluster = covered < coverage * doc.page_count and len(reps) < max_reps
        cluster_reps = [pages[len(pages) // 2]] if is_rep_cluster else []
        if is_rep_cluster and len(pages) > 50:  # very dominant layout: show two examples
            cluster_reps.append(pages[len(pages) // 4])
        reps.extend(cluster_reps)
        covered += len(pages)
        clusters.append({
            "signature": list(map(str, sig)),
            "n_pages": len(pages),
            "pages": pages,  # full list - validation uses it to localize failures per cluster
            "representatives": sorted(cluster_reps),
        })
    # first two pages carry format identity (title, TOC) - always include as context
    for p in (0, 1):
        if p < doc.page_count and p not in reps:
            reps.append(p)
    return {"clusters": clusters, "page_lines": page_lines, "representatives": sorted(set(reps))}


def dump_rep_page(lines: list[Line], path: str) -> None:
    """Structured text dump of one page - this is what recipe induction gets to see."""
    with open(path, "w", encoding="utf-8") as f:
        for L in lines:
            color = "#{:06x}".format(L.colors[-1]) if L.non_black else "black  "
            f.write(f"x={L.x0:6.1f} y={L.y0:6.1f} sz={L.size:4.1f} {color} {'B' if L.bold else ' '} | {L.text}\n")


def list_root_pdfs() -> list[str]:
    return sorted(
        os.path.join(CRF_DIR, fn)
        for fn in os.listdir(CRF_DIR)
        if fn.lower().endswith(".pdf")
    )


def doc_key(path: str) -> str:
    """Filesystem-safe per-document key. Long names get a hash suffix so two
    documents sharing a 70-char prefix cannot collide on the same output dir
    (which would silently cross-contaminate clusters.json and extraction CSVs).
    A stem with no ASCII alphanumerics at all (fully non-Latin filenames)
    sanitizes to bare underscores - every such file would collide on '_', so
    those become a hash key outright. Partially non-Latin names can still
    collide after sanitization; both batch drivers guard that with a loud
    doc_key-collision check before spending any budget."""
    stem = os.path.splitext(os.path.basename(path))[0]
    key = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem)
    if not re.search(r"[A-Za-z0-9]", key):
        return "doc_" + hashlib.sha1(stem.encode("utf-8")).hexdigest()[:12]
    if len(key) <= 70:
        return key
    return key[:61] + "_" + hashlib.sha1(stem.encode("utf-8")).hexdigest()[:8]
''')


In [ ]:
# --- pipeline module: generic_profile.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('generic_profile.py', r'''"""Generic structural page profile + similarity-threshold clustering.

The SHIPPED stage-0 front-end (stage0_cluster.py and the Dataiku notebook call
cluster_pages_generic). It replaced the five-signal fingerprint in common.py -
which remains available as the v1 reference for comparison probes - and is
built so a reviewer cannot call any part of it CRF-specific:

  * every line becomes one TOKEN describing pure typography/geometry -
    (x-band, size-vs-modal, bold, color, character-class). No regex for
    brackets, no "field number" notion, no named convention of any kind.
  * a page is the SET of its line tokens (presence, not counts: per-page
    count jitter is content, not template - measured in
    count_transform_sweep.py).
  * tokens are ubiquity-damped PER DOCUMENT: weight(t) = 1 - (df/n)^3.
    A token printed on every page is the document's own template chrome -
    headers, footers, boilerplate - and carries no template-discriminating
    information, so it weighs 0. Discovered from the document itself, never
    assumed. (Diagnosed on the QSC eSource book, where the 17 most-ubiquitous
    tokens - 11 of them printed on all 609 pages - carried 85-94% of every
    page's mass and fused all templates above any threshold - see
    qsc_merge_diag.py.)
  * page similarity = weighted Jaccard - the standard near-duplicate measure
    (Broder's shingling + TF-IDF-style weighting; MinHash approximates it at
    web scale, at <=1000 pages we compute it exactly, keeping determinism
    and zero dependencies).
  * clustering = best-fit leader clustering (canopy-style) in a canonical
    content-derived page order, a centroid-level merge pass, and one
    centroid-reassignment sweep. Deterministic and page-order-invariant,
    O(n*k), stdlib only.

The knob count drops from five hand-picked signals to ONE threshold (theta) -
and theta itself is NOT shipped as a constant: select_theta() picks it per
document by persistence/stability selection (cluster at a grid of sensible
values, find where the partition stops changing, take the middle of the widest
stable plateau). A fixed value would be corpus-fit by construction; a value
derived from the document being processed cannot be. The labeled page pairs in
theta_sanity_sweep.py are used only to TEST the result, never to set the knob.

Hardcoding audit: generalization_audit.py proves the properties hardcoding
would violate - word-scramble invariance (profiles bit-identical when every
word on every page is replaced), damping-exponent plateau (not a knife-edge
value, and per-document theta selection absorbs most of its effect),
cross-book mixture purity (chrome discovery adapts, no cross-contamination),
page-order invariance (canonical processing order), graceful tiny-document
degradation.
"""
from __future__ import annotations

import math
from collections import Counter, defaultdict

from common import Line, build_page_lines

X_BINS = 6
# Fixed-theta fallback for component tests and probes. The shipped path is
# select_theta(): per-document stability selection over THETA_GRID. (0.40 sits
# mid-plateau on every labeled invariant we have - theta_sanity_sweep.py - but
# any fixed constant is corpus-fit by construction, hence the selector.)
DEFAULT_THETA = 0.40
# Grid endpoints are structural properties of Jaccard similarity, not tuning:
# below 0.30 "same layout" would mean sharing under a third of weighted
# structural mass (meaninglessly loose); above 0.60 it would demand a
# near-identity that per-page content jitter never allows.
THETA_GRID = (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60)


def _charclass(text: str) -> str:
    """Character-composition class - typography only, no domain patterns."""
    t = text.strip()
    if not t:
        return "sym"
    letters = sum(c.isalpha() for c in t)
    digits = sum(c.isdigit() for c in t)
    if letters == 0 and digits > 0:
        return "num"        # "3", "19.0", "12/04/2024"
    if letters == 0:
        return "sym"        # "____", "---", "|"
    if t.upper() == t.lower():
        # caseless scripts (CJK, Thai, Arabic, Hebrew...): "ALL CAPS" is
        # vacuously true and CJK also prints no word spaces, so the Latin
        # code heuristic below would tag EVERY text line as code and erase
        # this token dimension document-wide. Split word/prose by word count
        # where spaces exist, else by length (a short run is a label, a long
        # one a sentence; 12 chars ~ 6+ ideographs - a coarseness knob like
        # the others, not a correctness constraint).
        return "prose" if len(t.split()) >= 3 or len(t) >= 12 else "word"
    if " " not in t and (digits > 0 or t.upper() == t):
        return "code"       # "[AESEV]", "SAS_NAME", "F14" - spaceless letter+digit/caps runs
    words = len(t.split())
    if words <= 2:
        return "word"       # "Severity", "Collection Date"
    return "prose"          # full sentences / questions


def _size_rel(size: float, modal: float) -> str:
    if modal <= 0:
        return "="
    r = size / modal
    return "-" if r < 0.9 else "+" if r > 1.15 else "="


COUNT_CAP = 3  # for counts="capped" research mode


def page_profile(lines: list[Line], page_width: float, page_height: float,
                 counts: str = "presence") -> Counter:
    """Structural line tokens; the page's word-blind layout profile.

    Default is PRESENCE (which token kinds exist), not counts: measured across
    all 7 CRF books + OOD docs (count_transform_sweep.py), presence gives the
    fattest same-template stacks because per-page count jitter (3 fields vs 5)
    is content, not template. y-position is excluded for the same reason:
    vertical extent tracks content volume, not template identity - which is
    also why page_height is accepted (call-site symmetry) but unused."""
    if not lines:
        return Counter()
    sizes = Counter(round(L.size, 1) for L in lines)
    modal = sizes.most_common(1)[0][0]
    raw: Counter = Counter()
    for L in lines:
        # clamp both sides: cropbox quirks can put x0 slightly outside [0, width)
        xb = max(0, min(X_BINS - 1, int(L.x0 / max(page_width, 1) * X_BINS)))
        tok = (xb, _size_rel(L.size, modal),
               "B" if L.bold else ".", "C" if L.non_black else ".",
               _charclass(L.text))
        raw[tok] += 1
    if counts == "presence":
        return Counter(dict.fromkeys(raw, 1))
    return Counter({t: min(c, COUNT_CAP) for t, c in raw.items()})


def token_weights(profiles: dict[int, Counter]) -> dict:
    """Smooth ubiquity damping, discovered per document:

        weight(t) = 1 - (df/n)**3

    A token on every page (the document's own header/footer/boilerplate
    chrome) weighs 0; on half the pages 7/8; a rare token ~1. Nothing is
    assumed about WHAT chrome looks like - it is whatever this document
    repeats everywhere.

    Deliberately NOT classic log-IDF: log(n/df) over-rewards rare tokens,
    which on pages are content noise, and it splits same-template pages
    (measured: QSC p3/p10 and aCRF p5/p6 separated, coverage dropped on
    all 13 probe docs). The failure being corrected is ubiquity flooding
    (chrome = 85-94% of every QSC page's mass), so only ubiquity is damped;
    mid-frequency tokens - the per-template skeleton - keep full weight."""
    pages = [p for p in profiles.values() if p]
    n = len(pages)
    if not n:
        return {}
    df: Counter = Counter()
    for p in pages:
        for t in p:
            df[t] += 1
    return {t: 1.0 - (c / n) ** 3 for t, c in df.items()}


def weighted_jaccard(a: Counter, b: Counter, w: dict | None = None) -> float:
    """Jaccard over capped counts; with `w`, each token's contribution is
    scaled by its document IDF. Two non-empty pages whose entire mass is
    zero-weight chrome are structurally identical plain pages -> 1.0."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if w is None:
        inter = sum(min(a[t], b[t]) for t in a if t in b)
        union = sum(a.values()) + sum(b.values()) - inter
        return inter / union if union else 0.0
    # math.fsum: exact summation, so the result does not depend on token
    # iteration order (set order varies with per-process hash randomization)
    inter = math.fsum(w.get(t, 0.0) * min(a[t], b[t]) for t in a if t in b)
    union = math.fsum(w.get(t, 0.0) * max(a.get(t, 0), b.get(t, 0)) for t in a.keys() | b.keys())
    if union <= 1e-9:
        return 1.0
    return inter / union


def cluster_profiles(profiles: dict[int, Counter], theta: float = DEFAULT_THETA,
                     weights: dict | None = None) -> list[list[int]]:
    """Best-fit leader clustering + one leader merge pass. Returns page-index lists."""
    if weights is None:
        weights = token_weights(profiles)
    # canonical processing order - CONTENT-derived, not document order: densest
    # profile first (dense pages are the best anchors for their template),
    # ties broken by token content. Makes the partition independent of page
    # arrival order (audit T4); the page-index tie-break only orders pages
    # with bit-identical profiles, which land in one stack regardless.
    order = sorted((i for i in profiles),
                   key=lambda i: (-sum(profiles[i].values()),
                                  str(sorted((str(t), c) for t, c in profiles[i].items())),
                                  i))
    leaders: list[tuple[Counter, list[int]]] = []
    empty: list[int] = []
    for i in order:
        p = profiles[i]
        if not p:
            empty.append(i)
            continue
        best_j, best_c = 0.0, -1
        for ci, (lead, _members) in enumerate(leaders):
            j = weighted_jaccard(p, lead, weights)
            if j > best_j:
                best_j, best_c = j, ci
        if best_c >= 0 and best_j >= theta:
            leaders[best_c][1].append(i)
        else:
            leaders.append((p, [i]))

    # Merge groups on their CENTROIDS, not on anchor pages: after damping, a
    # single page's residual profile is small (a handful of weighted tokens),
    # so one noisy token can hold two halves of a template family apart, while
    # the family centroids - which average out per-page noise - are clearly
    # within theta (measured on QSC: anchor-vs-anchor 0.35, centroid-vs-
    # centroid 0.47 for the two halves of the activity-page family).
    def centroid_of(members: list[int]) -> dict:
        cent: Counter = Counter()
        for m in members:
            cent.update(profiles[m])
        k = len(members)
        return {t: v / k for t, v in cent.items()}

    groups = [members for _lead, members in leaders]
    cents = [centroid_of(m) for m in groups]
    parent = list(range(len(groups)))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a in range(len(groups)):
        for b in range(a + 1, len(groups)):
            if weighted_jaccard(cents[a], cents[b], weights) >= theta:
                parent[find(b)] = find(a)

    merged: dict[int, list[int]] = defaultdict(list)
    for ci, members in enumerate(groups):
        merged[find(ci)].extend(members)
    out = [sorted(v) for v in merged.values()]

    # One centroid-reassignment sweep: a page assigned while its stack was
    # still small can fit a (now fully formed) different stack better. Single
    # sweep, deterministic; a page keeps its cluster unless a strictly better
    # centroid also clears theta.
    centroids = [centroid_of(members) for members in out]
    home = {p: ci for ci, members in enumerate(out) for p in members}
    moved: dict[int, list[int]] = defaultdict(list)
    for i in sorted(home):
        p = profiles[i]
        own = home[i]
        # a singleton's own centroid is itself (J=1.0), which would pin it
        # forever; let it compete for a real stack from a clean slate
        best_j, best_c = (0.0, own) if len(out[own]) == 1 else \
            (weighted_jaccard(p, centroids[own], weights), own)
        for ci, cent in enumerate(centroids):
            if ci == own:
                continue
            j = weighted_jaccard(p, cent, weights)
            if j > best_j:
                best_j, best_c = j, ci
        moved[best_c if best_j >= theta else own].append(i)
    out = [sorted(v) for v in moved.values() if v]

    if empty:
        out.append(empty)
    return sorted(out, key=lambda c: (-len(c), c[0]))


def _rand_index(pa: dict[int, int], pb: dict[int, int]) -> float:
    """Rand index between two partitions given as page -> cluster-id maps."""
    pages = sorted(pa)
    n = len(pages)
    if n < 2:
        return 1.0
    cont: Counter = Counter((pa[p], pb[p]) for p in pages)
    sum_ij = sum(math.comb(v, 2) for v in cont.values())
    sum_a = sum(math.comb(v, 2) for v in Counter(pa[p] for p in pages).values())
    sum_b = sum(math.comb(v, 2) for v in Counter(pb[p] for p in pages).values())
    total = math.comb(n, 2)
    return (total + 2 * sum_ij - sum_a - sum_b) / total


# Two adjacent-theta partitions count as "the same" when their Rand index is
# >= this bar: on a multi-hundred-page book RI 0.97 means under ~3% of page
# PAIRS change relation - statistically identical. This is a meta-knob (it
# defines "unchanged"), not a similarity threshold on any document content.
PLATEAU_RI = 0.97


def select_theta(profiles: dict[int, Counter], weights: dict | None = None,
                 grid: tuple = THETA_GRID, return_clusters: bool = False):
    """Per-document theta by PLATEAU (persistence) SELECTION - no fixed
    constant to overfit.

    Cluster the document at every theta on the grid and mark each adjacent
    pair of grid points whose partitions are statistically identical
    (Rand index >= PLATEAU_RI). Maximal runs of marked pairs are the
    document's stable plateaus: regions where the partition reflects real
    structure in THIS document rather than the knob. Choose the middle of
    the longest plateau; on ties prefer the looser (lower-theta) one, because
    over-merge is the cheaper miss - it surfaces as a zero-record stack and
    the coverage-confirm/audit loop repairs it, while over-split silently
    spends representative slots. If no pair clears the bar (a genuinely
    unstable document), fall back to the lower end of the most-stable pair.

    Scoring runs over marked PAIRS, never single endpoints, so a saturated
    end of the grid cannot win on one lucky neighbor (a first version scored
    endpoints on their single adjacent RI and picked theta=0.60 on the QSC
    book, splitting a known template family - caught by audit T6).

    Deterministic, len(grid) clustering passes, zero labeled data. Cost note:
    on repetitive books a pass is milliseconds, but the worst case (every page
    a unique layout) is O(n^2) similarity work per pass - minutes, not ms, at
    ~1000 pages. Bounded and rare, but the cost is real on genuinely
    heterogeneous documents. Returns (theta, diagnostics), or with return_clusters=True
    (theta, diagnostics, clusters_at_theta) so the caller can reuse the
    already-computed partition instead of re-clustering."""
    if weights is None:
        weights = token_weights(profiles)
    parts = []
    parts_clusters = []
    for th in grid:
        clusters = cluster_profiles(profiles, th, weights)
        parts_clusters.append(clusters)
        parts.append({p: ci for ci, c in enumerate(clusters) for p in c})
    m = len(grid)
    adj = [_rand_index(parts[i], parts[i + 1]) for i in range(m - 1)]
    marked = [ri >= PLATEAU_RI for ri in adj]

    runs = []  # (start_pair, end_pair) inclusive, over marked pairs
    start = None
    for i, ok in enumerate(marked + [False]):
        if ok and start is None:
            start = i
        elif not ok and start is not None:
            runs.append((start, i - 1))
            start = None
    if runs:
        # pair run s..e spans grid indices s..e+1; longest run wins,
        # ties -> lower theta
        s, e = max(runs, key=lambda r: (r[1] - r[0], -r[0]))
        chosen = (s + e + 1) // 2
    else:
        chosen = max(range(m - 1), key=lambda i: adj[i])  # lower end of best pair
    diag = {"grid": list(grid), "n_stacks": [len(set(p.values())) for p in parts],
            "adjacent_rand": [round(x, 4) for x in adj], "marked": marked,
            # lists, not tuples: this dict goes into clusters.json verbatim and
            # must round-trip through JSON unchanged
            "runs": [list(r) for r in runs], "chosen_index": chosen}
    if return_clusters:
        return grid[chosen], diag, parts_clusters[chosen]
    return grid[chosen], diag


def pick_representatives(clusters: list[list[int]], page_count: int,
                         max_reps: int = 10, coverage: float = 0.95,
                         is_blank=None) -> dict:
    """EXACT mirror of the rep policy in common.cluster_pages, applied to any
    partition, so method comparisons share identical selection logic.

    is_blank (optional page predicate): clusters made entirely of blank pages
    never receive a representative - an empty dump is a prompt slot the LLM
    cannot learn from, so blank clusters must not consume one even when slots
    remain after all content clusters. (They still count toward coverage
    accounting; blank pages need no parser.) Without the predicate the policy
    is byte-identical to common.cluster_pages."""
    def _blank(c: list[int]) -> bool:
        return is_blank is not None and all(is_blank(p) for p in c)

    ordered = sorted(clusters, key=lambda c: (_blank(c), -len(c)))
    out_clusters, covered, reps = [], 0, []
    for pages in ordered:
        is_rep_cluster = (covered < coverage * page_count and len(reps) < max_reps
                          and not _blank(pages))
        cluster_reps = [pages[len(pages) // 2]] if is_rep_cluster else []
        if is_rep_cluster and len(pages) > 50:
            cluster_reps.append(pages[len(pages) // 4])
        reps.extend(cluster_reps)
        covered += len(pages)
        out_clusters.append({"n_pages": len(pages), "pages": pages,
                             "representatives": sorted(cluster_reps)})
    for p in (0, 1):
        if p < page_count and p not in reps:
            reps.append(p)
    return {"clusters": out_clusters, "representatives": sorted(set(reps))}


def _cluster_signature(pages: list[int], profiles: dict[int, Counter],
                       weights: dict, theta: float) -> list[str]:
    """Human-readable stand-in for the five-signal tuple: the cluster's most
    discriminative token (highest weight x member-fraction). Keeps the output
    shape a drop-in for consumers that read cluster['signature'][0] as a
    display header (stage0_cluster.py). Empty-page clusters mirror common's
    '<empty>' signature."""
    frac: Counter = Counter()
    k = 0
    for p in pages:
        prof = profiles[p]
        if not prof:
            continue
        k += 1
        for t in prof:
            frac[t] += 1
    if k == 0:
        return ["<empty>"]
    top, score = None, -1.0
    for t, c in sorted(frac.items(), key=lambda kv: str(kv[0])):
        s = weights.get(t, 0.0) * (c / k)
        if s > score:
            top, score = t, s
    return [f"top:{top}", f"leader-jaccard theta={theta}"]


def cluster_pages_generic(doc, theta: float | None = None,
                          max_reps: int = 10, coverage: float = 0.95,
                          page_lines: dict[int, list[Line]] | None = None) -> dict:
    """Drop-in shaped counterpart of common.cluster_pages using the generic
    profile. theta=None (the default) selects theta per document by stability;
    pass a float to pin it (tests/probes). Accepts pre-parsed page_lines to
    avoid re-reading the PDF; if supplied it MUST cover every page
    0..doc.page_count-1 (representatives and coverage are computed against the
    full document)."""
    if page_lines is None:
        page_lines = {i: build_page_lines(doc[i]) for i in range(doc.page_count)}
    else:
        missing = set(range(doc.page_count)) - set(page_lines)
        if missing:
            raise ValueError(f"page_lines must cover all {doc.page_count} pages; "
                             f"missing e.g. {sorted(missing)[:5]}")
    profiles = {i: page_profile(page_lines[i], doc[i].rect.width, doc[i].rect.height)
                for i in page_lines}
    weights = token_weights(profiles)
    theta_diag = None
    if theta is None:
        theta, theta_diag, clusters = select_theta(profiles, weights, return_clusters=True)
    else:
        clusters = cluster_profiles(profiles, theta, weights)
    res = pick_representatives(clusters, doc.page_count, max_reps, coverage,
                               is_blank=lambda p: not profiles[p])
    for c in res["clusters"]:
        c["signature"] = _cluster_signature(c["pages"], profiles, weights, theta)
    res["page_lines"] = page_lines
    res["theta"] = theta
    if theta_diag is not None:
        res["theta_selection"] = theta_diag
    return res
''')


In [ ]:
# --- pipeline module: replay.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('replay.py', r'''"""LEGACY COMPARISON PATH - not the production flow.

Production induction is codegen.py (the LLM writes the extraction program itself;
no fixed engine catalog). This module remains for two reasons only:
  - FieldRec / ReplayResult are the shared record/result types
  - the engine catalog + reference_recipes serve as a measurable baseline the
    codegen output is compared against
Engine parameter DEFAULTS below (e.g. oid_header="Include", column headers
"Name"/"Export Name") come from the local sample corpus - they are baseline
fixtures, NOT priors to ship. Do not route production documents through here.

A recipe is a small JSON document emitted once per document by the induction LLM,
which only ever sees the stage-0 representative pages. The engines below execute
recipes at native speed - no LLM anywhere in the per-page path.

Any CRF layout is expressed as one of these engines plus parameters:

  adjacent_annotation - machine codes printed on their own annotation lines next to
                        (usually under) the human field label, in the same column
  anchored_blocks     - repeated anchor rows (e.g. bold 'Prefix: Activity #n' with a
                        number in a far column) delimit blocks; codes found per block
  numbered_join       - two page types joined by a printed number: content pages
                        (label + number) and definition pages (number + code column)
  column_table        - definition tables with a header row; label and code are
                        cells in configurable columns
  line_pattern        - generic fallback: regex roles over sequential text lines
"""
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass, field

import fitz

from common import Line, build_page_lines


@dataclass
class FieldRec:
    form_name: str
    field_name: str
    field_oid: str | None
    oid_alt: str | None = None
    page: int = 0
    extra: str = ""


@dataclass
class ReplayResult:
    format_id: str
    records: list[FieldRec] = field(default_factory=list)
    pages_with_fields: int = 0
    pages_total: int = 0
    definition_pages_seen: int = 0  # numbered_join only: pages matching def markers
    covered_pages: set = field(default_factory=set)  # 0-based, PRE-dedup (dedup keeps first
    # occurrence only, which would make repeated-field pages look uncovered)

    def dedup(self) -> None:
        """Collapse duplicates WITHIN a page only. The key includes `page`:
        a field repeated across pages (visit forms, log lines) is a real
        occurrence on every page it appears on. A page-blind key would keep
        only the first occurrence, making every later page look unextracted -
        the grounded audit then files phantom 'missed' findings against pages
        whose records were silently dropped, and convergence on repetitive
        form books (the core document class) becomes impossible."""
        seen, out = set(), []
        for r in self.records:
            key = (r.form_name.strip().lower(), r.field_name.strip().lower(),
                   (r.field_oid or "").strip(), r.page)
            if key not in seen:
                seen.add(key)
                out.append(r)
        self.records = out


def _compile(rxs) -> list[re.Pattern]:
    return [re.compile(rx) for rx in (rxs or [])]


def _is_noise(text: str, noise: list[re.Pattern]) -> bool:
    return any(rx.search(text) for rx in noise)


# --------------------------------------------------------------------------- #
# form-name strategies
# --------------------------------------------------------------------------- #
def find_form_name(lines: list[Line], cfg: dict, carry: str | None) -> str | None:
    strat = cfg.get("strategy")
    if strat == "regex":
        rx = re.compile(cfg["regex"])
        for L in lines:
            m = rx.match(L.text)
            if m:
                return m.group(1).strip()
        return carry if cfg.get("carry_forward") else None
    elif strat in ("colored_font", "font"):
        noise = _compile(cfg.get("noise"))
        best = None
        for L in lines:
            if L.size < cfg.get("min_size", 14):
                continue
            if strat == "colored_font" and not L.non_black:
                continue
            if _is_noise(L.text, noise) or not re.search(r"[A-Za-z]", L.text):
                continue
            if cfg.get("pick") == "last_above_first_field":
                # NOTE (known limitation, legacy path): callers pass full-page
                # lines, so this returns the last matching line on the PAGE, not
                # the last one above the first field. Good enough as a baseline.
                best = L.text.strip()
            else:
                return L.text.strip()
        if best:
            return best
    return carry if cfg.get("carry_forward") else None


# --------------------------------------------------------------------------- #
# engine: adjacent_annotation (codes on annotation lines next to the label)
# --------------------------------------------------------------------------- #
def extract_adjacent_annotation(pages, recipe: dict, result: ReplayResult) -> None:
    fcfg = recipe["fields"]
    oid_rx = re.compile(fcfg["oid_regex"])
    alt_rx = re.compile(fcfg["alt_regex"]) if fcfg.get("alt_regex") else None
    noise = _compile(fcfg.get("label_noise"))
    x_max = fcfg.get("column_x_max", 320)
    need_color = fcfg.get("oid_color_non_black", False)
    join_dy = fcfg.get("annotation_group_dy", 16)
    label_max_dy = fcfg.get("label_max_dy", 160)

    carry = None
    for pno, lines in pages:
        col = [L for L in lines if L.x0 <= x_max]
        # group consecutive bracket-annotation lines
        groups, cur = [], []
        for L in col:
            is_annot = bool(re.match(r"^\[.+", L.text)) and (L.non_black or not need_color)
            if is_annot and (not cur or L.y0 - cur[-1].y0 <= join_dy * max(1, len(cur))):
                cur.append(L)
            else:
                if cur:
                    groups.append(cur)
                cur = [L] if is_annot else []
        if cur:
            groups.append(cur)

        form_cfg = recipe["form_name"]
        form = find_form_name(lines, form_cfg, carry)
        got_field = False
        for g in groups:
            oid = alt = None
            for L in g:
                m = oid_rx.match(L.text)
                if m and not oid:
                    oid = m.group(1)
                elif alt_rx:
                    m2 = alt_rx.match(L.text)
                    if m2 and not alt:
                        alt = m2.group(1)
            if fcfg.get("alt_is_primary") and alt:
                oid, alt = alt, oid
            if not oid:
                continue
            # label: nearest non-noise line above the group, same column
            top = g[0]
            cand = [L for L in col
                    if L.y1 <= top.y0 + 2 and top.y0 - L.y0 <= label_max_dy
                    and not re.match(r"^\[.+", L.text) and not _is_noise(L.text, noise)]
            if not cand:
                continue
            cand.sort(key=lambda L: L.y0)
            label_parts = [cand[-1].text]
            # pull preceding wrapped label lines
            k = len(cand) - 2
            while k >= 0 and cand[k + 1].y0 - cand[k].y0 <= join_dy and len(label_parts) < 4:
                label_parts.insert(0, cand[k].text)
                k -= 1
            label = re.sub(r"\s+", " ", " ".join(label_parts)).strip()
            result.records.append(FieldRec(form or "", label, oid, alt, pno + 1))
            got_field = True
        if got_field:
            result.pages_with_fields += 1
        if form:
            carry = form


# --------------------------------------------------------------------------- #
# engine: anchored_blocks (repeated anchor rows delimit per-field blocks)
# --------------------------------------------------------------------------- #
def extract_anchored_blocks(pages, recipe: dict, result: ReplayResult) -> None:
    fcfg = recipe["fields"]
    act_rx = re.compile(fcfg["activity_regex"])
    oid_rx = re.compile(fcfg["oid_regex"])
    ax_min, ax_max = fcfg.get("activity_x", [150, 200])
    line_no_x = fcfg.get("line_number_x_min", 460)

    for pno, lines in pages:
        blocks = []  # (y, form, field)
        for L in lines:
            if not (ax_min <= L.x0 <= ax_max and L.bold):
                continue
            same_row_num = any(abs(o.y0 - L.y0) < 4 and o.x0 >= line_no_x for o in lines)
            m = act_rx.match(L.text)
            if m and same_row_num:
                blocks.append((L.y0, m.group(1).strip(), m.group(2).strip()))
        if not blocks:
            continue
        result.pages_with_fields += 1
        bounds = [b[0] for b in blocks] + [10 ** 9]
        for i, (y, form, fieldname) in enumerate(blocks):
            oids, question = [], ""
            for L in lines:
                if not (y < L.y0 < bounds[i + 1]):
                    continue
                m = oid_rx.search(L.text)
                if m:
                    oids.append(m.group(1))
                elif not question and L.bold and ax_min <= L.x0 <= ax_max + 5:
                    question = L.text.strip()
            for oid in dict.fromkeys(oids) or [None]:
                result.records.append(FieldRec(form, fieldname, oid, None, pno + 1, extra=question))


# --------------------------------------------------------------------------- #
# engine: numbered_join (content pages joined to definition pages by a number)
# --------------------------------------------------------------------------- #
def extract_numbered_join(pages, recipe: dict, result: ReplayResult) -> None:
    fcfg = recipe["fields"]
    form_rx = re.compile(recipe["form_name"]["regex"])
    def_marker = _compile(fcfg["definition_page_all"])
    num_rx = re.compile(fcfg.get("data_number_regex", r"^(\d+)(\.\d+)?$"))
    noise = _compile(fcfg.get("label_noise"))

    data: dict[str, list[tuple[str, str, int]]] = {}   # form -> [(num, label, page)]
    codes: dict[str, dict[str, str]] = {}              # form -> num -> code

    carry = None
    for pno, lines in pages:
        texts = [L.text for L in lines]
        is_def = all(any(rx.search(t) for t in texts) for rx in def_marker)

        form = None
        if not (is_def and fcfg.get("definition_form_from_carry")):
            for L in lines:
                m = form_rx.match(L.text)
                if m:
                    form = m.group(1).strip()
                    break
        if not form and (recipe["form_name"].get("carry_forward") or fcfg.get("definition_form_from_carry")):
            form = carry
        if not form:
            continue

        if is_def:
            result.definition_pages_seen += 1
            _parse_definition_page(lines, fcfg, num_rx, codes.setdefault(form, {}))
        else:
            rows = _parse_content_page(lines, fcfg, num_rx, noise)
            if rows:
                data.setdefault(form, []).extend((n, lbl, pno + 1) for n, lbl in rows)
                result.pages_with_fields += 1
            carry = form  # only content pages define which form a def page belongs to

    for form, rows in data.items():
        cmap = codes.get(form, {})
        for num, label, pno in rows:
            result.records.append(FieldRec(form, label, cmap.get(num), None, pno))


def _parse_definition_page(lines: list[Line], fcfg: dict, num_rx: re.Pattern, out: dict[str, str]) -> None:
    """Rows are keyed by the join number in the left margin (same regex as content
    pages, group 1 = key); the code lives in the column under `oid_header`."""
    anchor = next((L for L in lines if L.text.strip().startswith(fcfg.get("oid_header", "Include"))), None)
    if anchor is None:
        return
    code_x = anchor.x0 - 6
    # right-bound the code column at the next column header on the same header row,
    # otherwise neighbouring cells (e.g. a 'Type' column) get glued onto the code
    right = [L.x0 for L in lines if abs(L.y0 - anchor.y0) < 5 and L.x0 > anchor.x0 + 10]
    code_x_hi = min(right) - 4 if right else 10 ** 9
    num_x_max = fcfg.get("row_number_x_max", 108)
    rows = []
    for L in lines:
        t = L.text.strip()
        # anchors: the recipe's join-key regex, a bare integer, or an integer glued
        # to the first cell fragment on the same extracted line ("10 CSS0203A_")
        m = num_rx.match(t) or re.match(r"^(\d+)$", t) or re.match(r"^(\d+)\s+\S+$", t)
        if m and L.x0 < num_x_max:
            rows.append((L, m.group(1)))
    rows.sort(key=lambda t: t[0].y0)
    header_y = anchor.y0
    gaps = [rows[i + 1][0].y0 - rows[i][0].y0 for i in range(len(rows) - 1)]
    gaps = sorted(g for g in gaps if g > 0)
    typ_gap = gaps[len(gaps) // 2] if gaps else 40.0
    for i, (R, key) in enumerate(rows):
        y_lo = R.y0 - 8
        # bound the last row too, or the page footer gets swept into its code
        y_hi = rows[i + 1][0].y0 - 8 if i + 1 < len(rows) else R.y0 + max(3 * typ_gap, 48)
        parts = [L.text.strip() for L in lines
                 if code_x <= L.x0 < code_x_hi and y_lo <= L.y0 < y_hi and L.y0 > header_y + 5
                 and not re.search(r"\s", L.text.strip())]  # code fragments never contain spaces
        code = "".join(parts).strip()   # codes wrap across lines in narrow columns
        if code:
            out[key] = code


def _parse_content_page(lines, fcfg, num_rx, noise) -> list[tuple[str, str]]:
    num_x_min = fcfg.get("data_number_x_min", 480)
    label_x_max = fcfg.get("data_label_x_max", 320)
    embedded = bool(fcfg.get("number_embedded_in_label"))
    out = []
    for N in lines:
        t = N.text.strip()
        m = num_rx.match(t) if not embedded else num_rx.search(t)
        if not m or N.x0 < num_x_min:
            continue
        if embedded:
            # the join key can sit inside/next to the label line itself
            label = num_rx.sub("", t).strip(" -:\u2013")
            if not re.search(r"[A-Za-z]", label) or _is_noise(label, noise):
                same_row = [L for L in lines
                            if L is not N and abs(L.y0 - N.y0) <= 6 and L.x0 < N.x0
                            and re.search(r"[A-Za-z]", L.text) and not _is_noise(L.text, noise)]
                label = same_row[-1].text.strip() if same_row else ""
            if label:
                out.append((m.group(1), label))
            continue
        cand = [L for L in lines
                if L.x0 <= label_x_max and -8 <= L.y0 - N.y0 <= 16
                and re.search(r"[A-Za-z]", L.text) and not _is_noise(L.text, noise)]
        if cand:
            label = max(cand, key=lambda L: L.y0).text.strip()
            out.append((m.group(1), label))
    return out


# --------------------------------------------------------------------------- #
# engine: column_table (definition tables with a header row)
# --------------------------------------------------------------------------- #
def extract_column_table(pages, recipe: dict, result: ReplayResult) -> None:
    fcfg = recipe["fields"]
    def_marker = re.compile(fcfg["definition_page_regex"])
    row_rx = re.compile(fcfg.get("row_key_regex", r"^\[(\d+)\]$"))
    carry = None

    for pno, lines in pages:
        form_cfg = recipe["form_name"]
        form = find_form_name(lines, form_cfg, None)
        if form:
            carry = form
        if not any(def_marker.match(L.text.strip()) for L in lines):
            continue
        headers = {L.text.strip(): L.x0 for L in lines if L.bold}
        x_name = headers.get(fcfg.get("name_header", "Name"))
        x_oid = headers.get(fcfg.get("oid_header", "Export Name"))
        x_type = headers.get(fcfg.get("type_header", "Type"))
        if x_name is None or x_oid is None:
            continue
        rows = [L for L in lines if row_rx.match(L.text.strip()) and L.x0 < x_name - 5]
        rows.sort(key=lambda L: L.y0)
        got = False
        for i, R in enumerate(rows):
            y_lo, y_hi = R.y0 - 3, (rows[i + 1].y0 - 3 if i + 1 < len(rows) else 10 ** 9)
            names = [L.text.strip() for L in lines if abs(L.x0 - x_name) < 8 and y_lo <= L.y0 < y_hi]
            oid_cands = [L.text.strip() for L in lines
                         if abs(L.x0 - x_oid) < 8 and y_lo <= L.y0 < y_hi
                         and (x_type is None or L.x0 < x_type - 5)]
            if not names or not oid_cands:
                continue
            label = re.sub(r"\s+", " ", " ".join(names)).strip()
            result.records.append(FieldRec(carry or "", label, oid_cands[0], None, pno + 1))
            got = True
        if got:
            result.pages_with_fields += 1


# --------------------------------------------------------------------------- #
# engine: line_pattern (generic fallback for layouts none of the above fit)
# --------------------------------------------------------------------------- #
def extract_line_pattern(pages, recipe: dict, result: ReplayResult) -> None:
    fcfg = recipe["fields"]
    code_rx = re.compile(fcfg["code_regex"])
    label_rx = re.compile(fcfg.get("label_regex", r"^(?=.*[A-Za-z]).{3,140}$"))
    window = fcfg.get("label_window_lines", 4)
    noise = _compile(fcfg.get("label_noise"))
    carry = None
    for pno, lines in pages:
        form = find_form_name(lines, recipe["form_name"], carry)
        if form:
            carry = form
        got = False
        for i, L in enumerate(lines):
            t = L.text.strip()
            m = code_rx.search(t) if fcfg.get("code_search") else code_rx.match(t)
            if not m:
                continue
            label = None
            for j in range(i - 1, max(-1, i - 1 - window), -1):
                cand = lines[j].text.strip()
                if code_rx.search(cand) or _is_noise(cand, noise):
                    continue
                if label_rx.match(cand):
                    label = cand
                    break
            if label:
                result.records.append(FieldRec(form or "", label, m.group(1), None, pno + 1))
                got = True
        if got:
            result.pages_with_fields += 1


ENGINES = {
    "adjacent_annotation": extract_adjacent_annotation,
    "anchored_blocks": extract_anchored_blocks,
    "numbered_join": extract_numbered_join,
    "column_table": extract_column_table,
    "line_pattern": extract_line_pattern,
}


def build_pages(doc, recipe: dict, page_indices=None):
    skip = _compile(recipe.get("skip_page_if"))
    pages = []
    indices = range(doc.page_count) if page_indices is None else page_indices
    for i in indices:
        if i >= doc.page_count:
            continue
        lines = build_page_lines(doc[i])
        if skip and any(_is_noise(L.text, skip) for L in lines[:6]):
            continue
        pages.append((i, lines))
    return pages


def replay(pdf_path: str, recipe: dict, page_indices=None) -> ReplayResult:
    """Execute a recipe. `page_indices` restricts the run (used by the induction
    validator, which replays only the representative pages)."""
    doc = fitz.open(pdf_path)
    pages = build_pages(doc, recipe, page_indices)
    result = ReplayResult(format_id=recipe.get("format_id", "unknown"), pages_total=doc.page_count)
    engine = recipe.get("fields", {}).get("engine")
    if engine not in ENGINES:
        raise ValueError(f"unknown engine {engine!r}; must be one of {sorted(ENGINES)}")
    ENGINES[engine](pages, recipe, result)
    result.dedup()
    doc.close()
    return result


def detect_format(pdf_path: str, recipes: list[dict], sample_pages: list[int]) -> tuple[dict | None, dict]:
    """Score each recipe's detect markers against the sampled (representative) pages."""
    doc = fitz.open(pdf_path)
    text = "\n".join(doc[p].get_text() for p in sample_pages if p < doc.page_count)
    doc.close()
    scores = {}
    best, best_score = None, 0.0
    for r in recipes:
        det = r.get("detect", {})
        hits = sum(1 for rx in det.get("all", []) if re.search(rx, text))
        need = len(det.get("all", []) or [1])
        score = hits / need
        scores[r["format_id"]] = round(score, 2)
        if score > best_score or (score == best_score and best is None):
            if hits == need:
                best, best_score = r, score
    return best, scores
''')


In [ ]:
# --- pipeline module: induction.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('induction.py', r'''"""Stage 1 shared scoring/gates + the LEGACY recipe-induction path.

PRODUCTION-INTENT parts of this module: load_rep_pages, score, gate_problems,
gate_warnings (codegen.py imports them). The recipe/engine-catalog prompt and
induce_recipe below are the LEGACY comparison path - production induction is
codegen.py, where the LLM writes the extraction program itself.

The ONLY prior knowledge is: "this is a CRF". The LLM never gets a list of known
vendor formats. It sees the stage-0 representative pages (structured text dumps
with geometry + font info, optionally page images) and must emit a recipe JSON
that parameterises one of the generic layout engines in replay.py.

Loop (all bounded, all artifacts saved):
  1. build induction prompt from representative pages
  2. LLM -> recipe JSON
  3. validate: replay the recipe over the representative pages only, compute
     quality metrics, and check them against acceptance gates
  4. if gates fail: send the metrics + a sample of the (bad) output back to the
     LLM for ONE revision round (2 attempts total by default)
  5. if still failing: mark document as needs-manual-template (fail loudly)

LLM transport is pluggable:
  - Dataiku LLM Mesh (production; see notebook)
  - local HTTP/CLI shims for testing
"""
from __future__ import annotations

import json
import os
import re

from replay import ENGINES, ReplayResult, replay

PROMPT_TEMPLATE = """You are configuring a deterministic PDF extraction engine for a clinical Case Report Form (CRF) document. You will see a small sample of REPRESENTATIVE PAGES (one or two per page-layout cluster) from a document that is {n_pages} pages long. The whole document is repetitive: what you see is representative of everything.

Each page is given as structured text lines with geometry:
    x=<left> y=<top> sz=<font-size> <color> <B if bold> | <text>

# Your task

Produce a JSON "recipe" that tells the engine how to extract, for EVERY field on EVERY page of the document:
  - form_name   : the name of the CRF form/section the field belongs to
  - field_name  : the human-readable field label/question
  - field_oid   : the machine identifier (OID / SAS name / export name / variable
                  code) if the document prints one; null if the document is not
                  annotated with machine codes

# Engines you can parameterise (pick exactly one)

1. "adjacent_annotation" - machine codes are printed on their own annotation lines
   (often bracketed, often colored) directly next to/under the field label, in the
   same column. Params: column_x_max, oid_regex (capture group 1 = code),
   alt_regex/alt_is_primary (a second code on nearby lines), oid_color_non_black,
   annotation_group_dy, label_max_dy, label_noise (regexes for lines that are
   never labels).
2. "anchored_blocks" - each field starts at a repeated anchor row (e.g. a bold
   'Prefix: Activity #1' line with a number in a far-right column); everything
   until the next anchor belongs to that field. Params: activity_regex (group 1 =
   form, group 2 = field), activity_x [min,max], line_number_x_min, oid_regex.
3. "numbered_join" - two page types per form: content pages print each label with
   a join number, definition pages map the same numbers to codes in a column
   under a header. Params: definition_page_all (regexes that all appear on
   definition pages), oid_header (text of the column header the codes sit under),
   row_number_x_max (join keys on definition pages must start left of this),
   data_number_regex (group 1 = join key; same regex applies on BOTH page types),
   data_number_x_min, data_label_x_max, label_noise,
   number_embedded_in_label (true when the join key is printed on/next to the
   label line itself, e.g. 'Consent Date  [2]', rather than in a separate far
   column; the engine then searches the regex inside lines right of
   data_number_x_min and takes the remaining text on that row as the label),
   definition_form_from_carry (true when definition pages do NOT print the form
   name; they are then attributed to the most recent content-page form).
   form_name.strategy must be "regex" for this engine (carry_forward supported).
4. "column_table" - definition tables with an explicit header row; label and code
   are cells of configurable columns. Params: definition_page_regex, row_key_regex,
   name_header, oid_header, type_header.
5. "line_pattern" - fallback: a code regex over lines, label = nearest previous
   line matching label_regex within label_window_lines. Params: code_regex,
   code_search (true = search inside line), label_regex, label_window_lines,
   label_noise.

# form_name strategies

- {{"strategy":"regex","regex":"^Form:\\\\s*(.+)$"}} - a header line matches a regex (group 1 = name)
- {{"strategy":"colored_font","min_size":14,"carry_forward":true,"noise":[...]}} - the form title is the big colored heading; carry_forward repeats the last seen title on continuation pages
- {{"strategy":"font","min_size":14,"carry_forward":true,"noise":[...]}} - same but title is not colored
- for engine anchored_blocks the form name comes from activity_regex group 1 automatically

# Output format (JSON only, no prose)

{{
  "format_id": "<short slug you invent>",
  "reasoning": "<2-4 sentences: what layout you saw and why you chose the engine>",
  "detect": {{"all": ["<2-3 regexes that identify this layout>"]}},
  "skip_page_if": ["<optional regexes: pages to skip entirely (title/TOC/approval)>"],
  "form_name": {{...}},
  "fields": {{"engine": "<one of the 5>", ...params...}}
}}

# Rules

- Regexes are Python re syntax inside JSON strings (escape backslashes).
- Codes are machine identifiers like AESTDAT / QVAL_GENDOTH - short, uppercase,
  underscores/digits allowed. Human text, dates and option values are NOT codes.
- Prefer the most specific engine that fits; use line_pattern only if nothing fits.
- If the document prints NO machine codes at all, still extract form_name +
  field_name (choose the engine that best yields labels; oid_regex may then match
  nothing) and say so in "reasoning".
- Numbers like x/y/size in the dumps are points; use them to set column bounds.

# Representative pages

{pages}
"""

REVISION_TEMPLATE = """Your previous recipe was executed on the SAME representative pages. It did not pass the quality gates.

Previous recipe:
{recipe}

Execution metrics:
{metrics}

Sample of extracted records (form_name | field_name | field_oid):
{sample}

Problems to fix (in priority order):
{problems}

Emit a corrected recipe now. Same output format: JSON only, no prose. You may switch engine entirely.
"""


def load_rep_pages(outdir: str, max_chars_per_page: int = 3600) -> tuple[str, list[int]]:
    """Concatenate the stage-0 representative page dumps for the prompt."""
    with open(os.path.join(outdir, "clusters.json"), encoding="utf-8") as f:
        meta = json.load(f)
    reps = meta["representative_pages_1based"]
    blocks = []
    for p in reps:
        path = os.path.join(outdir, f"rep_p{p}.txt")
        with open(path, encoding="utf-8") as f:
            body = f.read()
        cluster_size = next((c["n_pages"] for c in meta["clusters"] if p - 1 in c.get("representatives", [])), 1)
        if len(body) > max_chars_per_page:
            body = body[:max_chars_per_page] + "\n<...page truncated...>\n"
        blocks.append(f"--- page {p} of {meta['pages']} (layout cluster covers ~{cluster_size} pages) ---\n{body}")
    return "\n".join(blocks), [p - 1 for p in reps]


def parse_recipe(raw: str) -> dict:
    """First complete JSON object in an LLM reply. Balanced parse (raw_decode) -
    a greedy `\\{.*\\}` regex would span from the first to the LAST brace and be
    corrupted by prose containing braces around the payload."""
    dec = json.JSONDecoder()
    idx = raw.find("{")
    while idx != -1:
        try:
            out, _ = dec.raw_decode(raw, idx)
            if isinstance(out, dict):
                return out
        except json.JSONDecodeError:
            pass
        idx = raw.find("{", idx + 1)
    raise ValueError("no JSON object in LLM reply")


# --------------------------------------------------------------------------- #
# quality gates - two tiers, both computed on the FULL-document run:
#   gate_problems  = contract blockers (program is effectively not working)
#   gate_warnings  = quality signals; they drive revision rounds but must never
#                    permanently reject a document (a legitimately form-dense or
#                    label-sparse CRF may violate them while being correct - the
#                    grounded audit round judges real quality per page)
# --------------------------------------------------------------------------- #
CODE_SHAPE = re.compile(r"^[A-Z][A-Z0-9_]{1,39}$")


def score(result: ReplayResult, engine: str) -> dict:
    recs = result.records
    n = len(recs)
    m = {
        "engine": engine,
        "records": n,
        "pages_total": result.pages_total,
        "pages_with_fields": result.pages_with_fields,
        "definition_pages_seen": result.definition_pages_seen,
        "forms_nonempty_pct": round(100 * sum(1 for r in recs if r.form_name.strip()) / n) if n else 0,
        # contract-level shape check only: has letters (any script/case), sane length,
        # not a single machine-code token. Floor is 2 chars: many CJK labels are two
        # characters (e.g. two-ideograph words); 1 char is still junk in any script.
        # The code-shape exclusion additionally requires a digit or underscore:
        # a bare ALL-CAPS word ("AGE", "SEVERITY") is a legitimate caps-styled
        # label in many books, and this is a warning metric - a false "junk"
        # verdict on caps-label documents costs pointless revision rounds.
        "labels_look_human_pct": round(100 * sum(
            1 for r in recs
            if re.search(r"[^\W\d_]", r.field_name) and 2 <= len(r.field_name) <= 200
            and not (CODE_SHAPE.match(r.field_name.strip())
                     and re.search(r"[\d_]", r.field_name))) / n) if n else 0,
        # oid metrics are LEGACY (recipe/engine path extracted OIDs; codegen scope
        # is form+field only, where these are always 0) - kept for the comparison path
        "oids_present_pct": round(100 * sum(1 for r in recs if r.field_oid) / n) if n else 0,
        "oids_look_like_codes_pct": round(100 * sum(
            1 for r in recs if r.field_oid and CODE_SHAPE.match(r.field_oid.strip())) /
            max(1, sum(1 for r in recs if r.field_oid))),
        "distinct_forms": len({r.form_name.strip().lower() for r in recs if r.form_name.strip()}),
    }
    # informational only - never gated (density is a property of the document,
    # not of extraction correctness)
    m["fields_per_form"] = round(n / max(1, m["distinct_forms"]), 1)
    m["forms_per_100_pages"] = round(100 * m["distinct_forms"] / max(1, result.pages_total), 1)
    return m


def gate_problems(m: dict) -> list[str]:
    """Contract blockers only: the program crashed upstream, or its output is so
    small it is effectively not extracting. Nothing here encodes corpus statistics.

    The <5-records floor applies only to documents of >=20 pages: on a big book
    it means the program is broken, but a 1-2 page CRF can legitimately carry
    3 fields total, and a hard gate would flag every version of a correct
    program as needs_manual_template. Tiny-document low volume is a warning
    (gate_warnings) so the audit still scrutinizes it."""
    problems = []
    if m["records"] == 0:
        problems.append("The program extracted ZERO records from the document.")
        return problems
    if m["records"] < 5 and m["pages_total"] >= 20:
        problems.append(f"Only {m['records']} records from a {m['pages_total']}-page document - the program is effectively not extracting.")
    if m["records"] and not m["pages_with_fields"]:
        # contract: every record carries page (int, 1-based). Without valid pages
        # the output cannot be page-audited or coverage-checked at all.
        problems.append("Records carry no valid 1-based `page` numbers - the `page` "
                        "field of every returned record must be the page the field "
                        "appears on.")
    return problems


def gate_warnings(m: dict) -> list[str]:
    """Quality signals fed back to the revision loop. A document may legitimately
    violate these (unlabeled forms, very short labels, dense form books), so after
    the revision budget is exhausted the best warning-only attempt is still
    accepted - final quality judgment belongs to the grounded audit."""
    warnings = []
    if 0 < m["records"] < 5 and m["pages_total"] < 20:
        warnings.append(f"Only {m['records']} records from this {m['pages_total']}-page "
                        "document - plausible for a document this small, but make sure "
                        "no fields were missed.")
    if m["forms_nonempty_pct"] < 70:
        warnings.append(f"form_name empty for {100 - m['forms_nonempty_pct']}% of records - if the document does print form/section names, fix the form_name strategy (consider carrying the last seen title forward).")
    if m["labels_look_human_pct"] < 60:
        warnings.append(f"Only {m['labels_look_human_pct']}% of field_name values look like human labels (they look like codes/dates/junk) - fix label selection.")
    if m["oids_present_pct"] > 0 and m["oids_look_like_codes_pct"] < 80:
        warnings.append(f"Extracted field_oid values often don't look like machine codes ({m['oids_look_like_codes_pct']}% ok) - tighten oid_regex.")
    if m["engine"] == "numbered_join" and m["definition_pages_seen"] > 0 and m["oids_present_pct"] < 10:
        warnings.append(
            f"{m['definition_pages_seen']} definition pages were detected but codes were joined to only "
            f"{m['oids_present_pct']}% of labels. "
            "The join is broken: check data_number_regex (group 1 must capture the SAME key on both page types), "
            "row_number_x_max, oid_header, and consider number_embedded_in_label:true (key printed on the label line) "
            "or definition_form_from_carry:true (definition pages don't print the form name; attribute them to the "
            "most recent content-page form).")
    if m["distinct_forms"] > max(10, m["records"] // 2):
        warnings.append(
            f"{m['distinct_forms']} distinct form_names for {m['records']} records - check whether form detection "
            "is picking up field labels or body text as form names; if so, make the form_name pattern stricter "
            "(font size / color / position) or carry the last seen title forward. If the document genuinely has "
            "this many forms, keep it as is.")
    return warnings


def validate_recipe(pdf_path: str, raw_reply: str) -> dict:
    """Parse a reply, replay it over the FULL document (fast - no LLM), and gate.
    Full-document validation matters: representative pages alone can miss a broken
    join (e.g. definition pages present but codes never attached)."""
    try:
        recipe = parse_recipe(raw_reply)
        result = replay(pdf_path, recipe)
        metrics = score(result, recipe.get("fields", {}).get("engine", "?"))
        # legacy path keeps the old strict behavior: warnings block acceptance too
        problems = gate_problems(metrics) + gate_warnings(metrics)
        sample = "\n".join(f"{r.form_name} | {r.field_name} | {r.field_oid}"
                           for r in result.records[:25]) or "(none)"
    except Exception as e:
        recipe, result = None, None
        metrics, problems, sample = {"error": str(e)}, [f"Recipe failed to execute: {e}"], "(none)"
    return {"recipe": recipe, "result": result, "metrics": metrics,
            "problems": problems, "sample": sample}


def build_revision_prompt(raw_reply: str, verdict: dict) -> str:
    return REVISION_TEMPLATE.format(
        recipe=json.dumps(verdict["recipe"], indent=1) if verdict["recipe"] else raw_reply[:2000],
        metrics=json.dumps(verdict["metrics"], indent=1),
        sample=verdict["sample"],
        problems="\n".join(f"- {p}" for p in verdict["problems"]),
    )


def induce_recipe(pdf_path: str, outdir: str, call_llm, max_attempts: int = 3) -> dict:
    """call_llm: fn(prompt:str) -> str. Returns dict with recipe/metrics/attempts."""
    import fitz
    pages_text, _ = load_rep_pages(outdir)
    n_pages = fitz.open(pdf_path).page_count
    prompt = PROMPT_TEMPLATE.format(n_pages=n_pages, pages=pages_text)

    trail = []
    for attempt in range(1, max_attempts + 1):
        raw = call_llm(prompt)
        verdict = validate_recipe(pdf_path, raw)
        trail.append({"attempt": attempt, "raw_reply": raw, "recipe": verdict["recipe"],
                      "metrics": verdict["metrics"], "problems": verdict["problems"]})
        if not verdict["problems"]:
            return {"status": "ok", "recipe": verdict["recipe"], "attempts": trail}
        prompt = build_revision_prompt(raw, verdict)
    return {"status": "needs_manual_template", "recipe": None, "attempts": trail}
''')


In [ ]:
# --- pipeline module: codegen.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('codegen.py', r'''"""Format-agnostic induction via CODE GENERATION - no strategy catalog, no few-shots.

Scope: form_name + field_name only. Printed machine codes are NOT extracted;
OID resolution happens downstream by name mapping against the rule library.

The LLM sees ONLY:
  - the task (extract form_name / field_name for every field)
  - the input data schema (Line objects with geometry/font attributes)
  - the output contract (function signature + record dict shape)
  - generic quality constraints (the same ones the gates check)
  - the stage-0 representative pages of THIS document

It must write the extraction program itself. Nothing in the prompt encodes layout
knowledge from any particular CRF vendor or sample. Generated code runs in a
separate killable process (sandbox_runner.py) and is only accepted if it passes
full-document contract gates; gate warnings and per-cluster coverage go back to
the LLM for a bounded revision loop.
"""
from __future__ import annotations

import collections
import json
import os
import re
import subprocess
import sys
import tempfile

import fitz

from common import build_page_lines
from induction import gate_problems, gate_warnings, load_rep_pages, score
from replay import FieldRec, ReplayResult

HERE = os.path.dirname(os.path.abspath(__file__))

CODEGEN_PROMPT = """You are writing a deterministic extraction program for one specific clinical Case Report Form (CRF) PDF document. Below you will find a small sample of REPRESENTATIVE PAGES (one or two per page-layout cluster) from a document that is {n_pages} pages long. The document is highly repetitive: the sampled pages cover its layouts, but the unsampled pages contain different content in the same layouts.

Each sampled page is shown as structured text lines with geometry:
    x=<left> y=<top> sz=<font-size> <color-hex or 'black'> <B if bold> | <text>

# Your task

Write a Python function that will run UNCHANGED over all {n_pages} pages and extract every data-entry field of the CRF:
  - form_name : the CRF form/section the field belongs to
  - field_name: the human-readable field label/question

That is the complete output. Some CRFs also print machine codes or technical
annotations near fields - do NOT return those (they are resolved downstream by a
separate system). You may still USE such markings as structural landmarks if that
helps you locate fields reliably.

# Runtime contract

def extract(pages):
    # pages: list of (page_index_0based, lines) tuples for the ENTIRE document, in
    #        page order. lines is a list of Line objects sorted by y, then x.
    #        NOTE: y-then-x order is NOT reading order on multi-column pages; if the
    #        layout has side-by-side columns, use x coordinates to separate them.
    #        For right-to-left scripts the within-row order is right-to-left, and
    #        vertical text yields an arbitrary line order - reconstruct reading
    #        order from the coordinates when the script needs it.
    # Line attributes:
    #   .text  (str, stripped visible text of one visual line)
    #   .x0 .y0 .x1 .y1  (floats; PDF points; origin = top-left of the page)
    #   .size  (float; font size in points; the largest span on the line)
    #   .bold  (bool)
    #   .non_black  (bool; True if any text on the line is printed in color)
    # returns: list of dicts, one per extracted field occurrence:
    #   {{"form_name": str, "field_name": str, "page": int_1based}}

# Hard constraints

- Pure computation only. These modules are already available: re, math, collections,
  itertools, functools, string, unicodedata, bisect, statistics, json. You may not
  import anything else, access files/network, or print.
- Deterministic, fast, simple: loops, regexes, coordinate arithmetic. It must
  process all {n_pages} pages in seconds.
- Generalize from STRUCTURE, not content. The unsampled pages contain questions,
  values and section names you have never seen. Never key your logic on specific
  question wording from the samples; key it on geometry (x positions, font sizes,
  color, boldness), on repeated marker/header patterns, and on the SHAPE of text
  (regexes over character classes). The document may be in any language.
- You may keep state across pages (the function receives the whole document) -
  e.g. a form name announced once may govern many following pages.

# Quality bar (your program's output is machine-checked before acceptance)

- It must extract from every page that carries fields, not just the sampled ones.
- form_name should be non-empty for the large majority of records. If the document
  genuinely prints no form/section names, use the best available section context;
  leave it empty only as a last resort.
- field_name values must be human-readable label text - not machine codes, bare
  numbers, dates, or page furniture (headers/footers/page numbers/legends).
- Answer OPTIONS are not fields: choice values (e.g. Yes / No / Unknown / list
  items - examples here are English, apply the concept in the document's language)
  belong to a field, they are not field_name records themselves.
- No duplicate records for the same (form_name, field_name) pair beyond what the
  document itself repeats.

# Reply format

Reply with ONLY Python source code (no prose outside code comments). Start with a
comment block (3-6 lines) stating what layout you observed in the samples and the
extraction strategy you chose. Then define extract(pages) plus any helpers.

# Representative pages of this document

{pages}
"""

CODE_REVISION_TEMPLATE = """Your extraction program was executed over the FULL document. It did not pass the quality gates.

Your previous program:
{code}

Execution metrics:
{metrics}

Sample of extracted records (pN: form_name | field_name):
{sample}

Problems to fix (in priority order):
{problems}
{cluster_feedback}
Rewrite the program now. Same reply format: Python source only, define extract(pages).
Where your program already works, EXTEND it rather than rewriting it - do not lose
coverage on pages that were extracting correctly. Different page layouts may need
different handling inside the same extract() function.
"""

CLUSTER_FEEDBACK_TEMPLATE = """
# Per-layout coverage

Pages of this document are grouped into layout clusters (pages whose structural
layout profiles are similar).
Coverage of your program per cluster (clusters your program extracted nothing or
little from are the ones to investigate):

{table}

# Sample pages from poorly-covered parts of the document (you have NOT seen these before)

If these pages contain data-entry fields, add handling for their layout. If they
genuinely carry no fields (title/instructions/legend pages), ignore them - zero
coverage there is correct.

{failing_pages}
"""

COVERAGE_CONFIRM_TEMPLATE = """You previously wrote the extraction program below for a clinical CRF PDF (task: extract form_name + field_name for every data-entry field; field labels only, never machine codes, answer options, or page furniture). It passed the aggregate quality gates, but substantial parts of the document produced ZERO records. Below are sample pages from those parts (you have not seen these pages before).

Your current program:
{code}

For each sampled page decide: does this layout carry data-entry fields your program is missing, or is it genuinely field-free (title/TOC/instructions/definitions-only pages)?

{cluster_feedback}

Reply with EXACTLY one of:
- the single line: CONFIRM_NO_FIELDS
  (meaning: all shown layouts are genuinely field-free; the program is complete), or
- the FULL updated Python program (same reply format: source only, define
  extract(pages)) that keeps existing behavior for covered layouts and ADDS
  handling for the missed ones.
"""

_FENCE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.S)


def extract_source(raw: str) -> str:
    """Accept both bare source and fenced code blocks."""
    blocks = _FENCE.findall(raw)
    return (max(blocks, key=len) if blocks else raw).strip()


def run_extractor(source: str, pdf_path: str, timeout_s: int = 300) -> ReplayResult:
    """Run generated code over the whole document in a SEPARATE process.

    The child (sandbox_runner.py) restricts the namespace; the process boundary is
    what makes a runaway program killable and keeps every run's module state fresh
    (an in-process thread can be neither killed nor isolated - in a long-lived
    notebook kernel that leaks CPU and cross-document state)."""
    src_file = tempfile.NamedTemporaryFile("w", suffix=".py", delete=False,
                                           encoding="utf-8", dir=HERE)
    try:
        src_file.write(source)
        src_file.close()
        proc = subprocess.Popen(
            [sys.executable, os.path.join(HERE, "sandbox_runner.py"), src_file.name, pdf_path],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=HERE)
        try:
            out, err = proc.communicate(timeout=timeout_s)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.communicate()
            raise TimeoutError(f"generated extractor exceeded {timeout_s}s") from None
    finally:
        try:
            os.unlink(src_file.name)
        except OSError:
            pass

    if proc.returncode != 0:
        raise RuntimeError("extractor process died "
                           f"(exit {proc.returncode}): {err.decode('utf-8', 'replace')[:800]}")
    try:
        payload = json.loads(out.decode("utf-8", "replace"))
    except json.JSONDecodeError:
        raise RuntimeError(f"extractor process produced no valid output: {out[:400]!r}")
    if "error" in payload:
        raise RuntimeError(f"generated extractor crashed:\n{payload['error']}")

    raw = payload["records"]
    n_pages = payload["n_pages"]
    result = ReplayResult(format_id="codegen", pages_total=n_pages)
    malformed = 0
    for item in raw:
        if item is None or not str(item.get("field_name", "")).strip():
            malformed += 1
            continue
        try:
            page = int(item.get("page") or 0)
        except (TypeError, ValueError):
            page = 0
        # out-of-range pages are invalid, not "extra coverage": they would
        # inflate pages_with_fields (poisoning version_score / the improves()
        # coverage guard) and crash the audit's doc[p - 1] page lookup
        if not 1 <= page <= n_pages:
            page = 0
        result.records.append(FieldRec(
            form_name=str(item.get("form_name") or "").strip(),
            field_name=str(item["field_name"]).strip(),
            field_oid=None,  # out of extraction scope; resolved downstream by name
            page=page,
        ))
    result.covered_pages = {r.page - 1 for r in result.records if r.page}
    result.pages_with_fields = len(result.covered_pages)
    result.dedup()
    if malformed > max(5, len(raw) // 10):
        raise ValueError(f"{malformed} of {len(raw)} returned records were malformed "
                         "(not a dict, or empty field_name)")
    return result


def build_codegen_prompt(pdf_path: str, outdir: str) -> str:
    pages_text, _ = load_rep_pages(outdir)
    doc = fitz.open(pdf_path)
    n_pages = doc.page_count
    doc.close()
    return CODEGEN_PROMPT.format(n_pages=n_pages, pages=pages_text)


# --------------------------------------------------------------------------- #
# per-cluster / per-page localization of coverage holes
# --------------------------------------------------------------------------- #
def _load_cluster_meta(outdir: str) -> dict:
    with open(os.path.join(outdir, "clusters.json"), encoding="utf-8") as f:
        return json.load(f)


def cluster_stats(result, meta: dict) -> list[dict]:
    """Coverage of the extraction per layout cluster (0-based pages in meta).
    Uses PRE-dedup page coverage: dedup keeps only the first occurrence of a
    repeated field, which would make repetition-heavy clusters look uncovered."""
    stats = []
    covered_pages = result.covered_pages or {r.page - 1 for r in result.records if r.page}
    rec_pages = collections.Counter(r.page - 1 for r in result.records if r.page)
    for ci, c in enumerate(meta["clusters"]):
        pages = c["pages"]
        with_rec = [p for p in pages if p in covered_pages]
        stats.append({
            "cluster": ci,
            "n_pages": len(pages),
            "pages_with_records": len(with_rec),
            "coverage_pct": round(100 * len(with_rec) / max(1, len(pages))),
            "records": sum(rec_pages[p] for p in pages),
        })
    return stats


def weak_clusters(stats: list[dict], min_pages: int = 4, coverage_lt: int = 30) -> list[dict]:
    return sorted((s for s in stats if s["n_pages"] >= min_pages and s["coverage_pct"] < coverage_lt),
                  key=lambda s: -s["n_pages"])


def _dump_lines_text(lines, max_chars: int = 3200) -> str:
    buf = []
    for L in lines:
        color = "#{:06x}".format(L.colors[-1]) if L.non_black else "black  "
        buf.append(f"x={L.x0:6.1f} y={L.y0:6.1f} sz={L.size:4.1f} {color} {'B' if L.bold else ' '} | {L.text}")
    s = "\n".join(buf)
    return s[:max_chars] + ("\n<...page truncated...>" if len(s) > max_chars else "")


def _spread(items: list, k: int) -> list:
    """Deterministically pick up to k items spread across the list."""
    if len(items) <= k:
        return list(items)
    step = len(items) / k
    return [items[int(i * step)] for i in range(k)]


def build_cluster_feedback(pdf_path: str, weak: list[dict], meta: dict,
                           stats: list[dict], max_clusters: int = 2,
                           pages_per_cluster: int = 2) -> str:
    """Coverage table + dumps of previously-unshown pages from the weakest clusters."""
    if not weak:
        return ""
    shown = {p - 1 for p in meta["representative_pages_1based"]}
    table = "\n".join(
        f"  cluster {s['cluster']:>3}: {s['n_pages']:>4} pages, "
        f"{s['pages_with_records']:>4} with records ({s['coverage_pct']}%), {s['records']} records"
        for s in stats if s["n_pages"] >= 2)

    doc = fitz.open(pdf_path)
    sections = []
    for s in weak[:max_clusters]:
        candidates = [p for p in meta["clusters"][s["cluster"]]["pages"] if p not in shown]
        if not candidates:
            continue
        picks = sorted({candidates[len(candidates) // 3], candidates[(2 * len(candidates)) // 3]})
        for p in picks[:pages_per_cluster]:
            sections.append(f"--- page {p + 1} (cluster {s['cluster']}, {s['n_pages']} pages like this, "
                            f"{s['coverage_pct']}% covered) ---\n"
                            + _dump_lines_text(build_page_lines(doc[p])))
    doc.close()
    if not sections:
        return ""
    return CLUSTER_FEEDBACK_TEMPLATE.format(table=table, failing_pages="\n\n".join(sections))


def build_uncovered_feedback(pdf_path: str, result, meta: dict,
                             uncovered_pct_min: int = 40, max_pages: int = 4) -> str:
    """Fallback coverage signal for DEGENERATE clusterings (e.g. every page its own
    cluster): weak_clusters() only sees clusters of >=4 pages, so a document whose
    layout profiles fragment would never surface coverage holes. This samples
    uncovered pages directly, doc-wide, whenever a large share of pages produced
    nothing."""
    total = meta["pages"]
    covered = result.covered_pages or {r.page - 1 for r in result.records if r.page}
    uncovered = [p for p in range(total) if p not in covered]
    if 100 * len(uncovered) / max(1, total) < uncovered_pct_min:
        return ""
    shown = {p - 1 for p in meta["representative_pages_1based"]}
    candidates = [p for p in uncovered if p not in shown] or uncovered
    doc = fitz.open(pdf_path)
    sections = [f"--- page {p + 1} (uncovered) ---\n" + _dump_lines_text(build_page_lines(doc[p]))
                for p in _spread(candidates, max_pages)]
    doc.close()
    table = (f"  {len(uncovered)} of {total} pages ({round(100 * len(uncovered) / max(1, total))}%) "
             "produced no records (page layouts too fragmented for a per-cluster table)")
    return CLUSTER_FEEDBACK_TEMPLATE.format(table=table, failing_pages="\n\n".join(sections))


def validate_generated(pdf_path: str, raw_reply: str, outdir: str | None = None) -> dict:
    """Run + gate a generated program. Returns:
      problems  - contract blockers (crash / effectively-no-output): must be fixed
      warnings  - quality signals (form pct, label shape, form explosion): feed the
                  revision loop but never permanently reject a document by themselves
      cluster_feedback - coverage holes localized to clusters (or raw pages when the
                  clustering is degenerate), for revision/confirmation prompts"""
    source = extract_source(raw_reply)
    cluster_feedback, stats, weak = "", [], []
    try:
        result = run_extractor(source, pdf_path)
        metrics = score(result, "codegen")
        problems = gate_problems(metrics)
        warnings = gate_warnings(metrics)
        sample = "\n".join(f"p{r.page}: {r.form_name} | {r.field_name}"
                           for r in result.records[:25]) or "(none)"
    except Exception as e:  # noqa: BLE001 - every failure becomes revision feedback
        result = None
        metrics = {"error": str(e)}
        problems = [f"Program failed to run: {e}"]
        warnings = []
        sample = "(none)"
    # Feedback building is OUR harness, not the generated program: its failures
    # (corrupt clusters.json, unreadable PDF page) must not masquerade as
    # "Program failed to run" and misdirect the revision at working code.
    if result is not None and outdir:
        try:
            meta = _load_cluster_meta(outdir)
            stats = cluster_stats(result, meta)
            weak = weak_clusters(stats)
            metrics["pages_covered_pct"] = round(
                100 * len(result.covered_pages) / max(1, meta["pages"]))
            if weak:
                cluster_feedback = build_cluster_feedback(pdf_path, weak, meta, stats)
            if not cluster_feedback:
                # the weak-cluster builder returns "" when every candidate page
                # was already shown; real doc-wide holes must still surface
                cluster_feedback = build_uncovered_feedback(pdf_path, result, meta)
        except Exception as e:  # noqa: BLE001
            print(f"    (coverage-feedback build failed, continuing without it: {e})")
            cluster_feedback, stats, weak = "", [], []
    return {"source": source, "result": result, "metrics": metrics,
            "problems": problems, "warnings": warnings, "sample": sample,
            "cluster_stats": stats, "weak_clusters": weak,
            "cluster_feedback": cluster_feedback}


# --------------------------------------------------------------------------- #
# iteration scoring: one comparable "result" per parser version so the loop
# controller can tell convergence from diminishing returns
# --------------------------------------------------------------------------- #
AUDIT_NOT_RUN = float("inf")


def version_score(verdict: dict, audit_issue_count: int | None) -> tuple:
    """Lexicographic quality of one parser version; LOWER is better.

    Order of importance:
      1. contract blockers (crash / effectively-no-output)  - dominate everything
      2. page-grounded audit issues                          - the real quality signal
      3. soft gate warnings (corpus-free shape priors)
      4. page coverage (negated)                             - tie-break only
    Audit counts are only comparable when produced on the SAME audit pages;
    the loop controller guarantees that by fixing the page sample once."""
    m = verdict.get("metrics") or {}
    return (len(verdict["problems"]),
            AUDIT_NOT_RUN if audit_issue_count is None else audit_issue_count,
            len(verdict.get("warnings") or []),
            -(m.get("pages_with_fields") or 0))


def improves(best_score: tuple | None, cand_score: tuple,
             best_cov: int = 0, cand_cov: int = 0,
             cov_floor: float = 0.9) -> bool:
    """Strict improvement over the best version so far.

    A candidate that loses more than 10% of covered pages is never an
    improvement, whatever its other numbers: audit issues are counted on a
    handful of pages, page coverage is doc-wide, and a 'fix' that silently
    drops whole layouts must not win on a lower issue count."""
    if best_score is None:
        return True
    if best_cov and cand_cov < cov_floor * best_cov:
        return False
    return cand_score < best_score


def _src_excerpt(source: str, cap: int = 12000) -> str:
    """The model is asked to EXTEND its own program; silently cutting the tail
    off makes it rewrite blind and lose coverage. Generated parsers routinely
    run 5-10 KB, so the cap is roomy - and when it does hit, the cut is
    announced instead of silent."""
    if len(source) <= cap:
        return source
    return (source[:cap]
            + f"\n# ... TRUNCATED: {len(source) - cap} more chars of your program "
              "are not shown; preserve the unshown logic when you rewrite ...")


def build_code_revision_prompt(verdict: dict) -> str:
    issues = ([f"- {p}" for p in verdict["problems"]]
              + [f"- (quality warning) {w}" for w in verdict.get("warnings", [])])
    return CODE_REVISION_TEMPLATE.format(
        code=_src_excerpt(verdict["source"]),
        metrics=json.dumps(verdict["metrics"], indent=1),
        sample=verdict["sample"],
        problems="\n".join(issues) or "- (see coverage feedback below)",
        cluster_feedback=verdict.get("cluster_feedback", ""),
    )


def build_coverage_confirm_prompt(verdict: dict) -> str:
    """For programs that PASS gates but leave whole clusters uncovered.
    Self-contained (includes the program) because transports are stateless."""
    return COVERAGE_CONFIRM_TEMPLATE.format(code=_src_excerpt(verdict["source"]),
                                            cluster_feedback=verdict["cluster_feedback"])


CONFIRM_TOKEN = "CONFIRM_NO_FIELDS"


def is_confirm_no_fields(reply: str) -> bool:
    """True only when the token IS the answer - alone on its own line and with no
    program in the reply. A substring test would misread prose that merely
    mentions the token ("I cannot CONFIRM_NO_FIELDS, here is the extension...")
    and silently discard the extension."""
    if "def extract" in reply:
        return False
    return any(line.strip() == CONFIRM_TOKEN for line in reply.splitlines())


# --------------------------------------------------------------------------- #
# grounded audit: judge quality against the document itself, never against
# corpus statistics (no assumptions about what a "typical" CRF looks like)
# --------------------------------------------------------------------------- #
AUDIT_PROMPT_TEMPLATE = """You are auditing the output of a deterministic extraction program that was run over a clinical CRF PDF. The program's task contract: for every data-entry field on every page, extract form_name (the CRF form/section the field belongs to) and field_name (the human-readable field label/question). Field labels only - machine codes, answer options (Yes/No/choice-list values), filled values, instructions, and page furniture (headers/footers/page numbers/legends) are NOT fields.

Below are {k} sampled pages of the document (structured text lines with geometry:
x=<left> y=<top> sz=<font-size> <color> <B if bold> | <text>), each followed by the
records the program extracted FROM THAT PAGE.

Audit each page strictly against what is printed on it:
- missed      : data-entry fields visible on the page that were not extracted
- false       : extracted records that are not actually data-entry fields
- wrong_form  : extracted records whose form_name does not match the form/section
                this page belongs to

Some sampled pages may have NO extracted records: if such a page prints data-entry
fields, list them under "missed"; if it is genuinely field-free (title/TOC/
instructions/definitions-only), return empty lists for it. If a page dump ends
with <...page truncated...>, audit only the shown region - records that belong
beyond the cut are not visible to you and must not be reported as "false".

# Reply format (JSON only, no prose)

[
 {{"page": <n>, "missed": ["<field label>", ...], "false": ["<field_name>", ...], "wrong_form": ["<field_name>", ...]}}
]

One object per audited page; empty lists mean that page's extraction is correct.

{pages_and_records}
"""


def pick_audit_pages(outdir: str, result: ReplayResult, max_pages: int = 6) -> list[int]:
    """Representative pages + covered NON-representative pages + up to two
    UNCOVERED pages (1-based).

    The non-rep covered picks test GENERALIZATION: representative pages were
    visible at induction time, so auditing only those would validate what the
    model already saw. The uncovered picks close two review blind spots: a
    wrong CONFIRM_NO_FIELDS verdict would otherwise never be re-examined, and
    coverage holes too small/diffuse for the confirm-round thresholds would
    never reach any reviewer. On an uncovered page the auditor either lists
    missed fields (driving a revision) or returns empty lists (independently
    confirming it field-free). Deterministic (spread picks, no RNG)."""
    meta = _load_cluster_meta(outdir)
    covered = sorted({r.page for r in result.records if r.page})
    covered_set = set(covered)
    uncovered = [p for p in range(1, meta["pages"] + 1) if p not in covered_set]
    unc_slots = min(2, max_pages // 3, len(uncovered))
    reps = set(meta["representative_pages_1based"])
    covered_reps = [p for p in covered if p in reps]
    covered_other = [p for p in covered if p not in reps]
    budget = max_pages - unc_slots
    half = budget // 2
    picks = _spread(covered_reps, half) + _spread(covered_other, budget - half)
    if len(picks) < budget:  # one pool was short - refill from the other
        rest = [p for p in covered if p not in picks]
        picks += _spread(rest, budget - len(picks))
    picks += _spread(uncovered, unc_slots)
    return sorted(set(picks))


def build_audit_prompt(pdf_path: str, outdir: str, result: ReplayResult,
                       max_pages: int = 6, pages: list[int] | None = None,
                       ) -> tuple[str, list[int]]:
    """Pair each sampled page dump with the records the program extracted from it.
    Pass `pages` to re-audit a fixed sample (issue counts are only comparable
    across programs when they are counted on the SAME pages)."""
    audit_pages = pages if pages is not None else pick_audit_pages(outdir, result, max_pages)
    by_page: dict[int, list] = collections.defaultdict(list)
    for r in result.records:
        by_page[r.page].append(r)
    doc = fitz.open(pdf_path)
    sections = []
    for p in audit_pages:
        recs = "\n".join(f"  extracted: {r.form_name} | {r.field_name}"
                         for r in by_page.get(p, [])) or "  (no records extracted from this page)"
        # roomier cap than the induction dumps: the auditor judges records
        # against the page, so cutting the page bottom would turn every record
        # from the hidden region into a phantom "false" finding
        sections.append(f"--- page {p} ---\n"
                        f"{_dump_lines_text(build_page_lines(doc[p - 1]), max_chars=6000)}\n\n"
                        f"Records the program extracted from page {p}:\n{recs}")
    doc.close()
    prompt = AUDIT_PROMPT_TEMPLATE.format(k=len(audit_pages),
                                          pages_and_records="\n\n".join(sections))
    return prompt, audit_pages


def parse_audit_reply(raw: str) -> list[dict]:
    """First complete JSON array in the reply (balanced parse, not greedy regex -
    prose containing brackets before/after the payload must not corrupt it)."""
    dec = json.JSONDecoder()
    idx = raw.find("[")
    while idx != -1:
        try:
            out, _ = dec.raw_decode(raw, idx)
            if isinstance(out, list):
                return out
        except json.JSONDecodeError:
            pass
        idx = raw.find("[", idx + 1)
    raise ValueError("no JSON array in audit reply")


def audit_issues(verdicts: list[dict]) -> int:
    return sum(len(v.get("missed") or []) + len(v.get("false") or []) + len(v.get("wrong_form") or [])
               for v in verdicts)


def audit_problem_lines(verdicts: list[dict]) -> list[str]:
    problems = []
    for v in verdicts:
        p = v.get("page")
        for kind, label in (("missed", "fields visible on the page but NOT extracted"),
                            ("false", "extracted records that are not data-entry fields"),
                            ("wrong_form", "records attributed to the wrong form")):
            items = v.get(kind) or []
            if items:
                shown = "; ".join(str(x) for x in items[:8])
                problems.append(f"Page {p}: {label}: {shown}"
                                + (f" (+{len(items) - 8} more)" if len(items) > 8 else ""))
    return problems
''')


In [ ]:
# --- pipeline module: sandbox_runner.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('sandbox_runner.py', r'''"""Child-process runner for LLM-generated extraction code.

Runs as its OWN interpreter process (spawned by codegen.run_extractor) so that:
  - a hung/looping program can be killed hard on timeout (a thread cannot),
  - every run gets fresh module objects (generated code that mutates re/collections
    cannot leak state into later attempts or other documents),
  - a memory-hungry program dies with the child, not with the notebook kernel.

usage: python sandbox_runner.py <source.py> <doc.pdf>
stdout: one JSON object, either {"records": [...], "n_pages": N} or {"error": "..."}.

THREAT MODEL - read before trusting this. The namespace restriction below is a
guard against ACCIDENTAL imports/IO by generated code; it is NOT a security
boundary (CPython exec offers none - object-graph walks reach os regardless).
And "the author is our own LLM" is only half the story: the LLM's INPUT is the
PDF's own text, so a hostile document can prompt-inject the code-writing round,
and whatever gets written runs here with the calling user's privileges and
network access. The process boundary gives kill-ability and state isolation,
not privilege isolation. For untrusted input sources, run the whole pipeline
(or at least this child) in an unprivileged, network-less container.
"""
import builtins
import json
import sys
import traceback

# Modules the generated program may use: pure computation + text handling.
# unicodedata/string matter for non-Latin documents; nothing here reaches IO.
import bisect
import collections
import functools
import itertools
import math
import re
import statistics
import string
import unicodedata

_ALLOWED_MODULES = {m.__name__: m for m in (
    re, math, collections, itertools, functools, string, unicodedata, bisect,
    statistics, json)}

# A terminating-but-runaway program (e.g. a cross-product bug) can emit records
# without bound; serializing gigabytes would blow up the PARENT - the exact
# failure the process boundary exists to contain. 200k records on a 1000-page
# document is 200 fields/page: far beyond any real form book.
MAX_RECORDS = 200_000

_SAFE_BUILTIN_NAMES = [
    "abs", "all", "any", "ascii", "bin", "bool", "bytearray", "bytes",
    "callable", "chr", "complex", "delattr", "dict", "dir", "divmod",
    "enumerate", "filter", "float", "format", "frozenset", "getattr", "hasattr",
    "hash", "hex", "id", "int", "isinstance", "issubclass", "iter", "len",
    "list", "locals", "globals", "map", "max", "memoryview", "min", "next",
    "object", "oct", "ord", "pow", "range", "repr", "reversed", "round", "set",
    "setattr", "slice", "sorted", "staticmethod", "classmethod", "property",
    "str", "sum", "super", "tuple", "type", "vars", "zip", "NotImplemented",
    # class statements compile to a __build_class__ call - without it any
    # generated program that defines a class dies with NameError
    "__build_class__",
    # exception names a legitimate program may raise or CATCH; a missing name
    # here turns a valid `except MemoryError:` into a NameError at runtime
    "ArithmeticError", "AssertionError", "AttributeError", "BaseException",
    "Exception", "GeneratorExit", "IndexError", "KeyError", "LookupError",
    "MemoryError", "NameError", "NotImplementedError", "OverflowError",
    "RecursionError", "RuntimeError", "StopAsyncIteration", "StopIteration",
    "TypeError", "UnicodeDecodeError", "UnicodeEncodeError", "UnicodeError",
    "ValueError", "ZeroDivisionError",
]


def _restricted_import(name, *args, **kwargs):
    root = name.split(".")[0]
    if root in _ALLOWED_MODULES:
        # delegate to the real import machinery so dotted forms work too:
        # `from collections.abc import Iterable` needs the submodule loaded,
        # which returning the bare root object cannot provide
        return builtins.__import__(name, *args, **kwargs)
    raise ImportError(f"module {name!r} is not available in the extraction sandbox")


def _sandbox_globals() -> dict:
    safe = {n: getattr(builtins, n) for n in _SAFE_BUILTIN_NAMES}
    safe["__import__"] = _restricted_import
    safe["print"] = lambda *a, **k: None
    g = {"__builtins__": safe, "__name__": "<generated_extractor>"}
    g.update(_ALLOWED_MODULES)
    return g


def _limit_memory() -> None:
    try:  # Linux/Dataiku only; Windows has no resource module
        import resource
        resource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))
    except Exception:  # noqa: BLE001
        pass


def main() -> None:
    _limit_memory()
    source_path, pdf_path = sys.argv[1], sys.argv[2]
    try:
        import fitz
        from common import build_page_lines

        with open(source_path, encoding="utf-8") as f:
            source = f.read()
        g = _sandbox_globals()
        exec(compile(source, "<generated_extractor>", "exec"), g)  # noqa: S102
        fn = g.get("extract")
        if not callable(fn):
            raise ValueError("generated code does not define extract(pages)")

        doc = fitz.open(pdf_path)
        if doc.needs_pass:
            raise ValueError("PDF is password-protected")
        pages = [(i, build_page_lines(doc[i])) for i in range(doc.page_count)]
        n_pages = doc.page_count
        doc.close()

        raw = fn(pages)
        if not isinstance(raw, list):
            try:  # a yield-based extract() is a legitimate program - materialize it
                raw = list(raw)
            except TypeError:
                raise ValueError(f"extract() must return a list, got {type(raw).__name__}")
        if len(raw) > MAX_RECORDS:
            raise ValueError(f"extract() returned {len(raw)} records "
                             f"(cap {MAX_RECORDS}) - runaway output")
        records = []
        for item in raw:
            if not isinstance(item, dict):
                records.append(None)  # malformed marker, judged by the parent
                continue
            try:  # page must be JSON-serializable here: a weird object (bytes,
                page = int(item.get("page"))  # ndarray) would kill json.dumps
            except (TypeError, ValueError):   # below and lose the whole verdict
                page = None
            records.append({"form_name": str(item.get("form_name") or ""),
                            "field_name": str(item.get("field_name") or ""),
                            "page": page})
        payload = {"records": records, "n_pages": n_pages}
    except Exception:  # noqa: BLE001 - everything becomes revision feedback upstream
        payload = {"error": traceback.format_exc(limit=6)}
    sys.stdout.write(json.dumps(payload))


if __name__ == "__main__":
    main()
''')


In [ ]:
# --- pipeline module: stage0_cluster.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('stage0_cluster.py', r'''"""Stage 0: cluster every page of every CRF by structural layout, save representative
page dumps (structured text) and PNGs. These representatives are all the FIRST
code-generation call gets to see; later loop rounds may additionally show the
model pages it under-covered (cluster feedback, coverage confirmation) and the
audit's sampled pages - always dumps produced by this same page model, never
raw PDF access.

Clustering front-end: generic_profile.cluster_pages_generic - word-blind typography
tokens per line, per-document ubiquity damping (chrome discovery), weighted-Jaccard
leader clustering, and a similarity threshold theta selected PER DOCUMENT by
stability. The five-signal fingerprint it replaced lives on in common.py as the v1
reference used by comparison probes.
"""
import glob
import json
import os

import fitz

from common import OUT_DIR, doc_key, dump_rep_page, list_root_pdfs
from generic_profile import cluster_pages_generic


def run(path: str) -> dict:
    """Cluster one document; returns meta. Documents the pipeline cannot process
    are detected HERE, before any LLM budget is spent, and marked with a non-ok
    status in the meta (drivers must check it): encrypted PDFs, and scanned/
    image-only PDFs with no text layer (OCR is out of scope - fail loudly)."""
    key = doc_key(path)
    out = os.path.join(OUT_DIR, key)
    os.makedirs(out, exist_ok=True)
    # remove stale outputs FIRST (incl. clusters.json): if this run crashes
    # mid-way, a leftover meta from a previous run would otherwise pass
    # doc_status() as 'ok' while its rep dumps are gone
    stale_meta = os.path.join(out, "clusters.json")
    if os.path.exists(stale_meta):
        os.remove(stale_meta)
    for stale in glob.glob(os.path.join(out, "rep_*")):
        os.remove(stale)
    doc = fitz.open(path)
    try:  # close in finally: a mid-run exception must not leak the handle
        # (on Windows an open handle keeps the staged PDF locked for re-runs)
        if doc.needs_pass or doc.page_count == 0:
            status = "encrypted" if doc.needs_pass else "no_pages"
            meta = {"file": os.path.basename(path), "status": status,
                    "pages": 0, "n_clusters": 0, "representative_pages_1based": [], "clusters": []}
            with open(os.path.join(out, "clusters.json"), "w", encoding="utf-8") as f:
                json.dump(meta, f, indent=1)
            return meta
        res = cluster_pages_generic(doc)
        clusters = res["clusters"]
        page_lines = res["page_lines"]

        pages_without_text = sum(1 for lines in page_lines.values() if not lines)
        text_layer_pct = round(100 * (doc.page_count - pages_without_text) / max(1, doc.page_count))

        rep_pages = res["representatives"]
        for p in rep_pages:
            dump_rep_page(page_lines[p], os.path.join(out, f"rep_p{p + 1}.txt"))
            doc[p].get_pixmap(dpi=100).save(os.path.join(out, f"rep_p{p + 1}.png"))

        meta = {
            "file": os.path.basename(path),
            # >=20% text pages -> proceed, but text_layer_pct travels with the
            # meta so drivers can surface partially scanned books (their scanned
            # pages are unreachable by design - OCR is out of scope)
            "status": "ok" if text_layer_pct >= 20 else "no_text_layer",
            "pages": doc.page_count,
            "text_layer_pct": text_layer_pct,
            "theta": res["theta"],  # per-document similarity threshold chosen by stability
            "theta_selection": res.get("theta_selection"),
            "n_clusters": len(clusters),
            "representative_pages_1based": [p + 1 for p in rep_pages],
            "clusters": [{k: v for k, v in c.items() if k != "signature"} | {"header": c["signature"][0]} for c in clusters],
        }
        with open(os.path.join(out, "clusters.json"), "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=1)
        return meta
    finally:
        doc.close()


if __name__ == "__main__":
    summary = []
    for path in list_root_pdfs():
        try:
            m = run(path)
        except Exception as e:  # same policy as the notebook cell: one corrupt
            print(f"{os.path.basename(path)[:60]:60s} stage0 FAILED: {e!r}")
            continue            # PDF must not sink the batch
        summary.append(m)
        flag = "" if m.get("status", "ok") == "ok" else f"  [{m['status']} - skipping induction]"
        theta = f" theta*={m['theta']:.2f}" if m.get("theta") is not None else ""
        print(f"{m['file'][:60]:60s} pages={m['pages']:5d} clusters={m['n_clusters']:3d} "
              f"reps={len(m['representative_pages_1based']):3d}{theta}{flag}")
    total_pages = sum(m["pages"] for m in summary)
    total_reps = sum(len(m["representative_pages_1based"]) for m in summary)
    print(f"\nTOTAL pages={total_pages}  representative pages={total_reps} "
          f"({100 * total_reps / total_pages:.1f}% would go to the LLM)")
''')


In [ ]:
# --- pipeline module: run_cli_induction.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('run_cli_induction.py', r'''"""Run the codegen induction loop against a REAL external model via the Cursor CLI.

This exists to validate the production model (Claude Sonnet 4.5) locally, before
the Dataiku notebook run. The CLI call is a plain chat completion: the prompt file
goes in on stdin, the reply comes out on stdout. The agent process runs in an
EMPTY sandbox directory outside the repo: its working dir exposes nothing. (A
print-mode agent does retain tool access to absolute paths, so the blind
property also rests on the prompt containing page dumps only - never repo or
artifact paths.)

Loop semantics (mirrors the intended Dataiku notebook). Every cycle produces ONE
parser version and ends with one comparable result:

  generate/revise -> full-document run + gates
                  -> [one-time coverage confirmation; an adopted extension is
                     scored as its OWN next version, after the pre-extension
                     program got its own trail entry and chance at best]
                  -> grounded audit on a page sample FIXED at the first audit
                  -> version_score = (hard problems, audit issues, warnings, -coverage)

Stopping (no phase-local budgets; one rule set for the whole loop):
  converged  gates pass, the audit finds zero issues, and the version is the new
             best -> accept immediately
  plateau    a version fails to strictly improve on the PREVIOUS one
             (diminishing returns; audit counts are compared on the same pages,
             and a version that loses >10% page coverage never counts as improved
             - neither for continuing the loop nor for best selection). Two
             CONSECUTIVE gate-failed versions never plateau: identical crash
             scores are not diminishing returns, they are zero returns - the
             revision loop keeps trying until the budget cap.
  budget     at most --max-versions cycles (default 5)
  error      transport/audit failure -> stop with what we have

A zero-issue audit is only TRUSTED when the reply actually covers every audited
page; a malformed or partial audit reply gets ONE reprompt before it counts as
failed/partial. Issue counts ignore pages outside the fixed sample. The BEST
version (not the last) is exported; if every version hard-failed the document is
flagged needs_manual_template. The controller is a small explicit state machine
on purpose - it maps 1:1 onto a LangGraph StateGraph (nodes: generate / validate
/ confirm / audit; conditional edges = the stop rules) if the Dataiku notebook
later wants checkpointing/tracing, with zero logic changes.

Usage:
  python run_cli_induction.py --model claude-4.5-sonnet [--only substring] [--max-versions 5]

Outputs per document (suffix keeps them separate from the parent-model runs):
  codegen_reply_<model>_<n>.py   raw model replies (one per version; an adopted
                                 coverage extension is also kept as _confirm.txt)
  codegen_trail_<model>.json     {stop_reason, versions, best_version, score_key,
                                 cycles}; an adopted extension appears as its own
                                 version right after the version it extends
  generated_extractor_<model>.py best accepted program
  fields_codegen_<model>.csv     full-document extraction
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import re
import shutil
import subprocess
import tempfile

from codegen import (AUDIT_NOT_RUN, audit_issues, audit_problem_lines,
                     build_audit_prompt, build_code_revision_prompt,
                     build_codegen_prompt, build_coverage_confirm_prompt,
                     improves, is_confirm_no_fields, parse_audit_reply,
                     run_extractor, validate_generated, version_score)
from common import OUT_DIR, doc_key, list_root_pdfs

DEFAULT_DOCS = [
    "384-201-00002_Annotated_Unique_CRF_04Nov2024",
    "QSC302573_Final_AnnotatedCRFs_16Oct2024-326-201-00007_1_",
    "MAC186_X11-201-00001_eCRF_v1.10_form_tracker_v1.6_06Mar2025",
]


def find_agent() -> str:
    for cand in (shutil.which("agent"),
                 os.path.join(os.environ.get("USERPROFILE", ""), ".local", "bin", "agent.exe"),
                 os.path.join(os.environ.get("USERPROFILE", ""), ".local", "bin", "agent")):
        if cand and os.path.exists(cand):
            return cand
    raise SystemExit("cursor CLI 'agent' not found on PATH or in ~/.local/bin")


def call_cli(agent_bin: str, model: str, prompt: str, timeout_s: int = 1200) -> str:
    sandbox = tempfile.mkdtemp(prefix="crf_llm_sandbox_")
    try:
        proc = subprocess.run(
            # --trust only trusts the EMPTY sandbox dir the process is started in
            [agent_bin, "-p", "--trust", "--model", model, "--output-format", "text"],
            input=prompt.encode("utf-8"),
            capture_output=True,
            timeout=timeout_s,
            cwd=sandbox,
        )
    finally:
        shutil.rmtree(sandbox, ignore_errors=True)
    out = proc.stdout.decode("utf-8", "replace")
    if proc.returncode != 0:
        err = proc.stderr.decode("utf-8", "replace")
        raise RuntimeError(f"agent CLI exited {proc.returncode}: {err[:1500]}")
    if not out.strip():
        raise RuntimeError(f"agent CLI returned empty reply; stderr: {proc.stderr.decode('utf-8', 'replace')[:800]}")
    return out


def slug(model: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", model.lower()).strip("_")


def doc_meta(outdir: str) -> dict:
    try:
        with open(os.path.join(outdir, "clusters.json"), encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):  # missing or corrupt stage-0 output
        return {"status": "missing_stage0"}


def doc_status(outdir: str) -> str:
    return doc_meta(outdir).get("status", "ok")


def save_reply(outdir: str, tag: str, name: str | int, reply: str, ext: str = "py") -> None:
    with open(os.path.join(outdir, f"codegen_reply_{tag}_{name}.{ext}"), "w", encoding="utf-8") as f:
        f.write(reply)


def jsonable_score(score: tuple) -> list:
    return ["not_audited" if s == AUDIT_NOT_RUN else s for s in score]


def coverage_of(verdict: dict) -> int:
    return (verdict.get("metrics") or {}).get("pages_with_fields", 0) or 0


def audit_page_num(v: dict) -> int | None:
    try:
        return int(v.get("page"))
    except (TypeError, ValueError):
        return None


def induce_document(call, model: str, tag: str, pdf: str, outdir: str,
                    initial_prompt: str, max_versions: int,
                    ) -> tuple[dict | None, list[dict], str, int]:
    """One document through the loop. Returns (best, trail, stop_reason, versions).

    `call` is the LLM transport: fn(prompt: str) -> reply str. The Cursor-CLI
    driver below and the Dataiku notebook (LLM Mesh) inject different transports
    around this SAME controller; `model` is a label for logs only.

    `best` is the best-scoring version so far (never the merely-latest one):
    {reply, verdict, score, version, audit_issues}. Plateau stopping compares
    CONSECUTIVE version scores; best selection additionally applies the
    coverage-regression guard in improves()."""
    trail: list[dict] = []
    best: dict | None = None
    prev: dict | None = None                # previous version's score/cov/had_problems
    audited_pages: list[int] | None = None  # fixed at the first audit forever
    confirm_pending = True                  # one coverage-confirm round per document
    prompt, kind = initial_prompt, "generate"
    versions, stop = 0, None

    def track(reply_, verdict_, score_, aud_):
        """Update best with a candidate version; returns whether it improved."""
        nonlocal best
        improved_ = improves(best["score"] if best else None, score_,
                             coverage_of(best["verdict"]) if best else 0,
                             coverage_of(verdict_))
        if improved_:
            best = {"reply": reply_, "verdict": verdict_, "score": score_,
                    "version": versions, "audit_issues": aud_}
        return improved_

    while stop is None:
        if versions >= max_versions:
            stop = "budget"
            break
        versions += 1
        print(f"    v{versions} [{kind}]: calling {model} ...", flush=True)
        try:
            reply = call(prompt)
        except Exception as e:  # noqa: BLE001
            trail.append({"version": versions, "kind": kind, "transport_error": str(e)})
            print(f"    transport error: {e}")
            stop = "transport_error"
            break
        save_reply(outdir, tag, versions, reply)
        verdict = validate_generated(pdf, reply, outdir)

        # ---- one-shot coverage confirmation, folded into this cycle ----------
        # The model either confirms the uncovered layouts are field-free or
        # extends its program. An adopted extension becomes a NEW version (it
        # costs budget) and is the program audited below - but the
        # PRE-extension version is scored and recorded FIRST: adoption only
        # checks record/coverage monotonicity, so an extension that is worse
        # on warnings must not silently erase a better program's claim to best.
        if (confirm_pending and not verdict["problems"] and verdict["cluster_feedback"]
                and versions < max_versions):
            confirm_pending = False
            wk = [(w["cluster"], w["n_pages"]) for w in verdict["weak_clusters"]]
            print(f"    coverage confirmation (uncovered: {wk or 'doc-wide holes'}) ...", flush=True)
            try:
                reply2 = call(build_coverage_confirm_prompt(verdict))
                save_reply(outdir, tag, "confirm", reply2, ext="txt")
                if is_confirm_no_fields(reply2):
                    trail.append({"version": versions, "kind": "confirm",
                                  "outcome": "confirmed_no_fields"})
                    print("    model confirmed uncovered clusters are field-free")
                else:
                    verdict2 = validate_generated(pdf, reply2, outdir)
                    # an extension may only ADD layouts: record count and page
                    # coverage must both hold (records alone can grow while whole
                    # covered layouts are dropped)
                    grew = (not verdict2["problems"]
                            and verdict2["metrics"].get("records", 0)
                            >= verdict["metrics"].get("records", 0)
                            and coverage_of(verdict2) >= coverage_of(verdict))
                    trail.append({"version": versions, "kind": "confirm",
                                  "metrics": verdict2["metrics"],
                                  "problems": verdict2["problems"],
                                  "outcome": "extended_program" if grew else "extension_rejected"})
                    if grew:
                        # close out the pre-extension version: own trail entry,
                        # own shot at best (un-audited -> AUDIT_NOT_RUN score)
                        orig_score = version_score(verdict, None)
                        orig_improved = track(reply, verdict, orig_score, None)
                        trail.append({"version": versions, "kind": kind,
                                      "metrics": verdict["metrics"],
                                      "problems": verdict["problems"],
                                      "warnings": verdict["warnings"],
                                      "audit_issues": None, "audit_pages": None,
                                      "audit_verdicts": None,
                                      "score": jsonable_score(orig_score),
                                      "became_best": orig_improved})
                        prev = {"score": orig_score, "cov": coverage_of(verdict),
                                "had_problems": False}
                        versions += 1
                        save_reply(outdir, tag, versions, reply2)
                        reply, verdict, kind = reply2, verdict2, "confirm_extension"
                        print(f"    extended program adopted as v{versions}: "
                              f"{json.dumps(verdict2['metrics'])}")
                    else:
                        print("    extension rejected (regression or gate failure); keeping current")
            except Exception as e:  # noqa: BLE001
                trail.append({"version": versions, "kind": "confirm", "transport_error": str(e)})
                print(f"    confirmation transport error: {e}")

        # ---- grounded audit (page sample fixed at the first audit) -----------
        # aud_count semantics: None = this version was NOT verified against pages
        # (no auditable pages / reply didn't cover the sample / audit errored).
        # None scores AUDIT_NOT_RUN, so an unverified version can never converge.
        # A malformed or page-skipping reply gets ONE reprompt before giving up:
        # a single bad completion must not end the whole document.
        aud_count, audit_verdicts, audit_partial = None, [], False
        if not verdict["problems"] and verdict.get("result"):
            try:
                aprompt, apages = build_audit_prompt(pdf, outdir, verdict["result"],
                                                     pages=audited_pages)
                if apages:
                    if audited_pages is None:
                        audited_pages = apages
                    sample = set(audited_pages)
                    for audit_try in (1, 2):
                        areply = call(aprompt)
                        try:
                            # only the fixed sample counts - issues reported for
                            # other pages would make version scores incomparable
                            audit_verdicts = [v for v in parse_audit_reply(areply)
                                              if audit_page_num(v) in sample]
                        except ValueError:
                            if audit_try == 1:
                                aprompt += ("\n\nYour previous reply contained no valid "
                                            "JSON array. Reply again with ONLY the JSON "
                                            "array described above, one object per "
                                            "audited page.")
                                continue
                            raise
                        if {audit_page_num(v) for v in audit_verdicts} == sample:
                            break
                        if audit_try == 1:
                            aprompt += ("\n\nYour previous reply skipped some audited "
                                        "pages. Reply again with ONLY the JSON array, "
                                        "exactly one object per page, for pages "
                                        + ", ".join(str(p) for p in audited_pages) + ".")
                    aud_count = audit_issues(audit_verdicts)
                    if {audit_page_num(v) for v in audit_verdicts} != sample:
                        audit_partial = True
                        if aud_count == 0:
                            aud_count = None  # zero by omission is not a clean audit
                            print(f"    audit reply skipped some of pages {audited_pages}; "
                                  "zero-issue count not trusted")
                        else:  # a partial nonzero count UNDERCOUNTS; keep it (it
                            # still drives the revision) but flag it in the trail
                            print(f"    audit: {aud_count} issue(s) on a PARTIAL reply "
                                  f"(pages {audited_pages})")
                    else:
                        print(f"    audit: {aud_count} issue(s) on pages {audited_pages}")
            except Exception as e:  # noqa: BLE001
                trail.append({"version": versions, "kind": "audit", "transport_error": str(e)})
                print(f"    audit error: {e}")
                stop = "audit_error"  # score this version un-audited, then stop

        # ---- score, track best, decide ---------------------------------------
        cand_score, cand_cov = version_score(verdict, aud_count), coverage_of(verdict)
        improved = track(reply, verdict, cand_score, aud_count)
        trail.append({"version": versions, "kind": kind,
                      "metrics": verdict["metrics"], "problems": verdict["problems"],
                      "warnings": verdict["warnings"], "audit_issues": aud_count,
                      "audit_pages": audited_pages if aud_count is not None else None,
                      "audit_partial": audit_partial or None,
                      "audit_verdicts": audit_verdicts or None,
                      "score": jsonable_score(cand_score), "became_best": improved})
        if verdict["problems"]:
            print(f"    problems: {verdict['problems']}")
        elif verdict["warnings"]:
            print(f"    warnings: {verdict['warnings']}")

        if stop:  # audit failed mid-cycle; version already scored and recorded
            break
        if not verdict["problems"] and aud_count == 0 and improved:
            # verified clean AND the new best - nothing left to iterate on.
            # (clean but NOT improved = it got there by dropping coverage; the
            # plateau rule below stops the loop and the earlier best is exported)
            stop = "converged"
            break
        if (prev is not None
                and not (prev["had_problems"] and bool(verdict["problems"]))
                and not improves(prev["score"], cand_score, prev["cov"], cand_cov)):
            # two consecutive GATE-FAILED versions are exempt: identical crash
            # scores are zero returns, not diminishing returns - keep revising
            # until the budget cap instead of giving up at version 2
            stop = "plateau"  # diminishing returns across two consecutive versions
            break
        prev = {"score": cand_score, "cov": cand_cov,
                "had_problems": bool(verdict["problems"])}

        # ---- next revision prompt --------------------------------------------
        if verdict["problems"]:
            prompt, kind = build_code_revision_prompt(verdict), "revise_gates"
        else:
            prompt = build_code_revision_prompt({
                "source": verdict["source"],
                "metrics": verdict["metrics"],
                "sample": verdict["sample"],
                "problems": ["(aggregate gates passed; a page-level audit of the document "
                             "found the issues below)"] + audit_problem_lines(audit_verdicts),
                "warnings": verdict["warnings"],
                "cluster_feedback": "",
            })
            kind = "revise_audit"

    return best, trail, stop or "budget", versions


def finalize_document(key: str, pdf: str, outdir: str, tag: str,
                      best: dict | None, trail: list[dict], stop_reason: str,
                      versions: int) -> dict:
    """Persist the trail, export the best program + full-document CSV, and build
    the summary row. Shared verbatim by this CLI driver and the Dataiku notebook."""
    with open(os.path.join(outdir, f"codegen_trail_{tag}.json"), "w", encoding="utf-8") as f:
        json.dump({"stop_reason": stop_reason, "versions": versions,
                   "best_version": best["version"] if best else None,
                   "score_key": ["gate_problems", "audit_issues",
                                 "warnings", "neg_pages_with_fields"],
                   "cycles": trail}, f, indent=1)

    row = {"doc": key, "versions": versions, "stop_reason": stop_reason}
    usable = best is not None and not best["verdict"]["problems"]
    if not usable:
        row["status"] = "needs_manual_template"
        return row

    verdict, aud = best["verdict"], best["audit_issues"]
    if aud is None:
        row["status"] = "ok_unaudited"     # never page-verified (audit error)
    elif aud > 0:
        row["status"] = "ok_audit_issues"  # best still has known page issues
    elif verdict["warnings"]:
        row["status"] = "ok_with_warnings"
    else:
        row["status"] = "ok"
    row["best_version"] = best["version"]
    source = verdict["source"]
    with open(os.path.join(outdir, f"generated_extractor_{tag}.py"), "w", encoding="utf-8") as f:
        f.write(source + "\n")
    try:
        full = run_extractor(source, pdf)
    except Exception as e:  # noqa: BLE001 - a flaky final replay must not
        row["status"] = "export_failed"    # abort the remaining documents
        row["error"] = str(e)
        return row
    with open(os.path.join(outdir, f"fields_codegen_{tag}.csv"), "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["form_name", "field_name", "page"])
        for r in full.records:
            w.writerow([r.form_name, r.field_name, r.page])
    row.update(fields=len(full.records),
               forms=len({r.form_name for r in full.records if r.form_name}),
               pages_with_fields=full.pages_with_fields,
               audit_issues=best["audit_issues"],
               warnings=verdict.get("warnings", []))
    return row


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--only", help="substring filter on doc key")
    ap.add_argument("--all-docs", action="store_true", help="run every doc, not just the default 3")
    ap.add_argument("--max-versions", type=int, default=5,
                    help="hard cap on parser versions per document (loop also stops "
                         "earlier on convergence or plateau)")
    args = ap.parse_args()
    if args.max_versions < 1:
        ap.error("--max-versions must be >= 1")
    agent_bin = find_agent()
    tag = slug(args.model)

    all_pdfs = list_root_pdfs()
    pdfs = {doc_key(p): p for p in all_pdfs}
    if len(pdfs) != len(all_pdfs):  # a collision would silently drop a document
        raise SystemExit("doc_key collision among input PDFs (near-identical names) "
                         "- rename the colliding files")
    docs = sorted(pdfs) if args.all_docs else [d for d in DEFAULT_DOCS if d in pdfs]
    if args.only:
        docs = [d for d in docs if args.only.lower() in d.lower()]

    summary = []
    for key in docs:
        pdf = pdfs[key]
        outdir = os.path.join(OUT_DIR, key)
        print(f"=== {key}")
        try:  # one bad document must not sink a multi-hour paid batch
            meta = doc_meta(outdir)
            status = meta.get("status", "ok")
            if status != "ok":
                # encrypted / scanned / zero-page / stage0 not run: no LLM budget spent
                summary.append({"doc": key, "status": f"skipped_{status}"})
                print(f"    skipped: {status}")
                continue
            # build the prompt fresh from the CURRENT stage-0 artifacts (and keep
            # a copy for inspection) - reading a pre-existing codegen_prompt.txt
            # would silently reuse stale rep dumps after a stage-0 rerun
            initial_prompt = build_codegen_prompt(pdf, outdir)
            with open(os.path.join(outdir, "codegen_prompt.txt"), "w", encoding="utf-8") as f:
                f.write(initial_prompt)

            transport = lambda prompt: call_cli(agent_bin, args.model, prompt)  # noqa: E731
            best, trail, stop_reason, versions = induce_document(
                transport, args.model, tag, pdf, outdir, initial_prompt, args.max_versions)
            row = finalize_document(key, pdf, outdir, tag, best, trail, stop_reason, versions)
            if meta.get("text_layer_pct", 100) < 100:
                # partially scanned book: those pages are unreachable by design
                row["text_layer_pct"] = meta["text_layer_pct"]
        except Exception as e:  # noqa: BLE001
            row = {"doc": key, "status": "error", "error": repr(e)}
        summary.append(row)
        print(f"    -> {row}")

    with open(os.path.join(OUT_DIR, f"cli_induction_summary_{tag}.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=1)
    print(json.dumps(summary, indent=1))


if __name__ == "__main__":
    main()
''')


In [ ]:
# --- pipeline module: oid_mapping.py (verbatim from the repo; regenerate, don't edit) ---
_write_module('oid_mapping.py', r'''"""Name -> OID mapping for non-Rave CRFs: the form-first funnel.

Annotated Rave books carry printed OIDs, so production never had a
name->OID step -- its rule matcher starts from OIDs it reads off the page
(review_table.fuzzy_match_fields). Non-Rave books yield only
(form_name, field_name), so this module reconstructs the missing OIDs
against the ecs_index library and hands them to that same downstream.

The funnel (each layer narrows, the last one judges):

  layer 1  FORM scoping    partial_ratio >= FORM_SCOPE_T (70) -- the exact
                           scorer/threshold production already uses to match
                           form names (review_table.get_standard_crf).
                           No form match -> unmapped.
  layer 2  FIELD in form   token_sort_ratio >= FIELD_T (85) against the
                           scoped rows only, best row per distinct OID ->
                           a shortlist of 1..MAX_CANDIDATES OIDs.
  layer 3  LLM ranker      EVERY shortlisted case is judged: pick one of the
                           listed OIDs or null. Single candidates are
                           confirm-or-null, not auto-accepted -- measured on
                           the Rave book, both lexical-layer errors were
                           lone candidates that passed the gate while the
                           right row was absent from the library ("Contact
                           Method" at similarity 100 from the WRONG
                           follow-up form). String scores generate
                           candidates; they never certify them.

Without an LLM (llm=None) the module runs deterministic-only: a single
candidate or a clear leader (>= MARGIN points) maps lexically, near-ties
abstain. That mode exists for offline smoke runs and A/B comparison.

Everything unresolved exits as unmapped, which is safe by design:
production writes LLM-generated rules for unmapped fields from their names
alone. The harmful failure is a *mismap* (silently attaches the wrong
form's library rules); every layer here is shaped to minimize that -- the
shortlist bound means the ranker can never introduce an OID, though it can
still echo a listed-but-wrong one, so mismap risk is reduced, not zero.

Inputs are pre-normalized strings (caller owns normalization); library
entries are dicts with keys: field, form (normalized), oid, and optionally
field_raw / form_raw for human-readable ranker prompts.
"""
import json
import re
from dataclasses import dataclass, field as dc_field

from rapidfuzz import fuzz

FORM_SCOPE_T = 70   # partial_ratio, mirrors review_table.get_standard_crf
FIELD_T = 85        # token_sort_ratio within the scoped form's rows
MARGIN = 5          # min lead over the best rival OID to map without the ranker
MAX_CANDIDATES = 5  # shortlist size handed to the ranker
RANKER_CHUNK = 40   # abstained cases per ranker call

MAPPED = "mapped"
ABSTAIN = "abstain"
UNMAPPED = "unmapped"


@dataclass
class MapResult:
    form: str
    field: str
    status: str                       # mapped | abstain | unmapped
    oid: str | None = None
    score: float = 0.0
    via: str = "lexical"              # lexical | ranker
    candidates: list = dc_field(default_factory=list)  # [(score, oid, label, form)]


def scope_form(form: str, lib: list[dict]) -> list[dict]:
    """Layer 1: only library rows whose form name matches the extracted one.

    Set semantics deliberately mirror production: get_standard_crf calls
    process.extract(..., score_cutoff=70, limit=len(all_form_names)) -- ALL
    forms above the cutoff, not the single best one. Related forms
    ("Adverse Events" / "Adverse Events - Serious") therefore co-exist in
    one scope; the ranker sees each candidate's library form and judges."""
    return [e for e in lib
            if fuzz.partial_ratio(form, e["form"]) >= FORM_SCOPE_T]


def map_field(form: str, field: str, lib: list[dict]) -> MapResult:
    """Layers 1-2, deterministic decision. Candidates are always attached so
    an LLM ranker can re-judge even lexically 'clear' picks (see map_pairs)."""
    if not form.strip() or not field.strip():
        return MapResult(form, field, UNMAPPED)
    scoped = scope_form(form, lib)
    if not scoped:
        return MapResult(form, field, UNMAPPED)

    # best-scoring row per distinct OID (Standard + Historical rows collapse)
    per_oid: dict[str, tuple[float, dict]] = {}
    for e in scoped:
        s = fuzz.token_sort_ratio(field, e["field"])
        if s < FIELD_T:
            continue
        key = e["oid"].upper()
        if key not in per_oid or s > per_oid[key][0]:
            per_oid[key] = (s, e)
    if not per_oid:
        return MapResult(form, field, UNMAPPED)

    ranked = sorted(((s, e) for s, e in per_oid.values()),
                    key=lambda t: t[0], reverse=True)
    best_s, best_e = ranked[0]
    shortlist = [(s, e["oid"], e.get("field_raw", e["field"]),
                  e.get("form_raw", e["form"]))
                 for s, e in ranked[:MAX_CANDIDATES]]
    if len(ranked) == 1 or best_s - ranked[1][0] >= MARGIN:
        return MapResult(form, field, MAPPED, oid=best_e["oid"], score=best_s,
                         candidates=shortlist)
    return MapResult(form, field, ABSTAIN, candidates=shortlist)


# --------------------------------------------------------------------------- #
# layer 3: the LLM ranker over abstained cases
# --------------------------------------------------------------------------- #
def build_ranker_prompt(cases: list[MapResult]) -> str:
    lines = [
        "You are resolving printed CRF field names to library OIDs.",
        "For each case, one or more library entries matched the printed",
        "field name by string similarity. String similarity generates these",
        "candidates but cannot certify them: a perfect label match can sit",
        "on the wrong form, and near-identical labels can be a canonical",
        "field vs its derived variant. Judge by MEANING in form context.",
        "",
        "Rules:",
        '- Reply with ONLY a JSON array: [{"case": 1, "oid": "..."}, ...]',
        "- \"oid\" must be exactly one of that case's listed OIDs, or null.",
        "- Candidates may come from DIFFERENT library forms (form matching",
        "  is fuzzy), and the printed form may not exist in the library at",
        "  all. If no candidate's library form is genuinely the printed",
        "  form, answer null -- however perfect the field label match is.",
        "- A single candidate is a question, not an answer: confirm the",
        "  library entry's field AND form genuinely mean the printed ones,",
        "  else null.",
        "- Prefer the OID representing direct entry of the printed field",
        "  over derived/computed variants, unless the label says otherwise.",
        "- When unsure, answer null. A wrong OID is worse than none: it",
        "  attaches the wrong form's validation rules.",
        "",
    ]
    for i, c in enumerate(cases, 1):
        lines.append(f"Case {i}:")
        lines.append(f"  form (as printed):  {c.form}")
        lines.append(f"  field (as printed): {c.field}")
        lines.append("  candidates:")
        for s, oid, label, lib_form in c.candidates:
            lines.append(f'    - {oid}  (library label: "{label}", '
                         f'library form: "{lib_form}", similarity {s:.0f})')
    return "\n".join(lines)


def _extract_json_array(reply: str) -> list | None:
    """First JSON array in the reply, tolerant of prose/markdown around it.

    A greedy first-'['-to-last-']' regex would break as soon as the prose
    contains any bracket, silently unmapping a whole chunk -- so parse the
    whole reply first, then raw_decode from each '[' until a list parses."""
    try:
        whole = json.loads(reply)
        if isinstance(whole, list):
            return whole
    except ValueError:
        pass
    dec = json.JSONDecoder()
    for m in re.finditer(r"\[", reply):
        try:
            val, _ = dec.raw_decode(reply, m.start())
        except ValueError:
            continue
        if isinstance(val, list):
            return val
    return None


def parse_ranker_reply(reply: str, cases: list[MapResult]) -> dict[int, str | None]:
    """case index (1-based) -> chosen OID (canonical library casing) or None.

    Anything malformed, out-of-range, or not on the case's shortlist is
    treated as None -- the ranker can never introduce an OID."""
    arr = _extract_json_array(reply)
    out: dict[int, str | None] = {}
    for item in arr if isinstance(arr, list) else []:
        if not isinstance(item, dict):
            continue
        try:
            idx = int(item.get("case"))
        except (TypeError, ValueError):
            continue
        if not (1 <= idx <= len(cases)):
            continue
        oid = item.get("oid")
        if oid is None:
            out[idx] = None
            continue
        allowed = {o.upper(): o for _, o, _, _ in cases[idx - 1].candidates}
        out[idx] = allowed.get(str(oid).strip().upper())
    return out


def rank_cases(cases: list[MapResult], llm) -> None:
    """Judge every candidate-bearing MapResult with the ranker.

    llm: callable prompt -> reply text (CLI locally, LLM Mesh in Dataiku).
    A case the ranker declines (or answers invalidly) becomes unmapped --
    including cases the lexical layer had marked mapped.

    All chunks are judged BEFORE any result is mutated: if the llm raises
    mid-batch, no case has changed state and the exception propagates, so a
    caller can never ship a half-ranked mixture of ranker and lexical
    verdicts (the caller decides whether to retry or fail the document)."""
    all_picks: list[tuple[MapResult, str | None]] = []
    for start in range(0, len(cases), RANKER_CHUNK):
        chunk = cases[start:start + RANKER_CHUNK]
        picks = parse_ranker_reply(llm(build_ranker_prompt(chunk)), chunk)
        all_picks.extend((c, picks.get(i)) for i, c in enumerate(chunk, 1))

    for c, oid in all_picks:
        if oid:
            # "confirm" only when the ranker kept the lexical leader; an
            # overridden leader is a pick, not a confirmation
            c.via = ("ranker-confirm" if c.status == MAPPED and c.oid
                     and oid.upper() == c.oid.upper() else "ranker-pick")
            c.status, c.oid = MAPPED, oid
        else:
            c.status, c.oid = UNMAPPED, None


def map_pairs(pairs: list[tuple[str, str]], lib: list[dict],
              llm=None) -> list[MapResult]:
    """Run the full funnel over (form, field) pairs.

    With an llm, EVERY pair that produced candidates is judged by the
    ranker (confirm-or-null for single candidates, pick-or-null for ties).
    Without one, lexical decisions stand and near-ties stay ABSTAIN."""
    results = [map_field(f, l, lib) for f, l in pairs]
    if llm is not None:
        judged = [r for r in results if r.candidates]
        if judged:
            rank_cases(judged, llm)
    return results
''')


In [ ]:
# Import the pipeline from the materialized modules. Pop-and-import (never
# importlib.reload): reload would re-execute a module from wherever it was FIRST
# imported, so a foreign 'codegen'/'common' already living in the kernel would
# silently shadow the bundle. Re-run safe top to bottom.
_PIPELINE_MODULES = ('common', 'generic_profile', 'replay', 'induction',
                     'codegen', 'stage0_cluster', 'run_cli_induction')
if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)
for _m in _PIPELINE_MODULES:
    sys.modules.pop(_m, None)

import common
import codegen
import stage0_cluster
import run_cli_induction as rci

for _m in _PIPELINE_MODULES:
    _f = os.path.abspath(getattr(sys.modules[_m], '__file__', '') or '')
    assert _f.startswith(os.path.abspath(MODULES_DIR) + os.sep),         _m + ' imported from ' + _f + ' instead of the bundle - check sys.path'

from common import CRF_DIR, OUT_DIR, doc_key, list_root_pdfs

os.makedirs(CRF_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print('work dir:', WORK)


In [ ]:
# Stage the input PDFs from the managed folder into the local work dir
# (PyMuPDF needs real files; managed folders may be S3-backed). PDFS - the list
# bound HERE - is what every later cell iterates: a warm work dir from an
# earlier session never adds documents the current DOC_FILTER excludes.
import re
import shutil

import dataiku

folder_in = dataiku.Folder(INPUT_FOLDER)
paths = sorted(p for p in folder_in.list_paths_in_partition() if p.lower().endswith('.pdf'))
if DOC_FILTER:
    paths = [p for p in paths if DOC_FILTER.lower() in p.lower()]
if MAX_DOCS:
    paths = paths[:MAX_DOCS]

flat = {}
for p in paths:
    local_name = re.sub(r'[\\/]+', '_', p.strip('/'))
    if local_name in flat:
        raise RuntimeError('two folder paths flatten to the same file name: '
                           + flat[local_name] + ' and ' + p + ' -> ' + local_name)
    flat[local_name] = p

PDFS = []
for local_name, p in sorted(flat.items()):
    dest = os.path.join(CRF_DIR, local_name)
    with folder_in.get_download_stream(p) as s, open(dest, 'wb') as f:
        shutil.copyfileobj(s, f)  # always overwrite - folder content may have changed
    PDFS.append(dest)

_keys = [doc_key(p) for p in PDFS]
assert len(set(_keys)) == len(_keys),     'doc_key collision among input PDFs (near-identical names) - rename the colliding files'
print(len(PDFS), 'pdf(s) staged')
for p in PDFS:
    print('  ', os.path.basename(p))


In [ ]:
# Stage 0 (pure Python, no LLM): cluster every page by structural layout and
# pick representative pages. The clusterer (generic_profile) is word-blind:
# each line becomes a typography/geometry token, tokens the document repeats
# on nearly every page (its own header/footer chrome) are discovered and
# down-weighted, pages are grouped by weighted-Jaccard similarity, and the
# similarity threshold theta is selected PER DOCUMENT by stability - no
# corpus-tuned constants. Encrypted or scanned (no text layer) PDFs are
# flagged here and never spend LLM budget. One corrupt PDF must not sink the
# batch: it is reported and skipped (no clusters.json -> the induction cell
# records it as skipped).
for _pdf in PDFS:
    try:
        _m = stage0_cluster.run(_pdf)
    except Exception as _e:
        print(f"{os.path.basename(_pdf)[:58]:58s} stage0 FAILED: {_e!r}")
        continue
    _flag = '' if _m.get('status') == 'ok' else '  [' + _m['status'] + ' - will skip induction]'
    _theta = ' theta*=%.2f' % _m['theta'] if _m.get('theta') is not None else ''
    print(f"{_m['file'][:58]:58s} pages={_m['pages']:5d} clusters={_m['n_clusters']:3d} "
          f"reps={len(_m['representative_pages_1based']):3d}{_theta}{_flag}")


In [ ]:
# LLM transport: Dataiku LLM Mesh (same get_llm/new_completion conventions as
# the ECS generation recipes). Plain-text completion per call, bounded retries.
import time

client = dataiku.api_client()
project = client.get_default_project()
llm_id = LLM_ID or project.get_variables()['local'].get('default_llm_model')
assert llm_id, 'set LLM_ID or the project variable default_llm_model'
llm = project.get_llm(llm_id)

def call_mesh(prompt, retries=2, backoff_s=15):
    last = None
    for attempt in range(retries + 1):
        try:
            comp = llm.new_completion()
            try:
                comp.settings.update(COMPLETION_SETTINGS)
            except Exception as e:
                # settings shape varies across mesh/provider versions; warn ONCE -
                # without maxOutputTokens the provider default cap may truncate
                # long generated programs (which then fail gates and burn budget)
                if not getattr(call_mesh, '_settings_warned', False):
                    call_mesh._settings_warned = True
                    print('WARNING: completion settings not applied (' + repr(e)
                          + '); mesh defaults in effect')
            comp.with_message(prompt)
            resp = comp.execute()
            if getattr(resp, 'success', False) and (resp.text or '').strip():
                return resp.text
            last = RuntimeError('unsuccessful or empty completion')
        except Exception as e:
            last = e
        if attempt < retries:
            time.sleep(backoff_s * (attempt + 1))
    raise RuntimeError('LLM Mesh call failed after ' + str(retries + 1) + ' attempts: ' + repr(last))

print('LLM Mesh id:', llm_id)


In [ ]:
# The induction loop. rci.induce_document is the SAME controller validated by
# stop_policy_test.py locally; only the transport (call_mesh) is Dataiku-specific.
# Stop rules: converged (clean audit) / plateau (no improvement between two
# consecutive versions) / budget (MAX_VERSIONS) - best version wins.
# A per-document failure becomes a summary row, never a lost batch (this can be
# a multi-hour paid run). Summary file is induction_summary_<tag>.json - the
# 'cli_' prefix is reserved for the local CLI driver so runs cannot be confused.
import json

tag = rci.slug(llm_id)
summary = []
for _pdf in PDFS:
    _key = doc_key(_pdf)
    _outdir = os.path.join(OUT_DIR, _key)
    print('===', _key)
    try:
        _meta = rci.doc_meta(_outdir)
        _status = _meta.get('status', 'ok')
        if _status != 'ok':
            summary.append({'doc': _key, 'status': 'skipped_' + _status})
            print('    skipped:', _status)
            continue
        _prompt = codegen.build_codegen_prompt(_pdf, _outdir)
        with open(os.path.join(_outdir, 'codegen_prompt.txt'), 'w', encoding='utf-8') as f:
            f.write(_prompt)
        _best, _trail, _stop, _versions = rci.induce_document(
            call_mesh, llm_id, tag, _pdf, _outdir, _prompt, MAX_VERSIONS)
        _row = rci.finalize_document(_key, _pdf, _outdir, tag, _best, _trail, _stop, _versions)
        if _meta.get('text_layer_pct', 100) < 100:
            # partially scanned book: its no-text pages are unreachable (OCR out
            # of scope) - surface that in the summary instead of hiding it
            _row['text_layer_pct'] = _meta['text_layer_pct']
    except Exception as _e:
        _row = {'doc': _key, 'status': 'error', 'error': repr(_e)}
    summary.append(_row)
    print('    ->', _row)

with open(os.path.join(OUT_DIR, 'induction_summary_' + tag + '.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=1)

import pandas as pd
pd.DataFrame(summary)


In [ ]:
# Persist all artifacts to the output managed folder, mirroring the local
# out/ tree: <doc_key>/clusters.json, rep_p*.txt, codegen_prompt.txt,
# codegen_reply_*<n>.py, codegen_trail_*.json, generated_extractor_*.py,
# fields_codegen_*.csv, plus induction_summary_*.json at the root.
# Scope: THIS run's documents only. A warm work dir may hold artifacts of
# documents an earlier session processed but the current DOC_FILTER excludes -
# re-uploading those would contradict the run scoping FETCH_CELL guarantees.
folder_out = dataiku.Folder(OUTPUT_FOLDER)
_run_keys = {doc_key(_p) for _p in PDFS}
uploaded = 0
for _root, _dirs, _files in os.walk(OUT_DIR):
    for _fn in _files:
        if _fn.endswith('.png') and not UPLOAD_PAGE_PNGS:
            continue
        _full = os.path.join(_root, _fn)
        _rel = os.path.relpath(_full, OUT_DIR).replace(os.sep, '/')
        _top = _rel.split('/')[0]
        if '/' in _rel and _top not in _run_keys:
            continue  # another session's document dir
        with open(_full, 'rb') as f:
            folder_out.upload_stream(_rel, f)
        uploaded += 1
print('uploaded', uploaded, 'file(s) to folder', OUTPUT_FOLDER,
      'for', len(_run_keys), 'document(s)')


In [ ]:
# OPTIONAL - name->OID mapping via the form-first funnel (oid_mapping.py):
#   1. form scoping   partial_ratio >= 70 (production's own convention,
#                     review_table.get_standard_crf)
#   2. field in form  token_sort_ratio >= 85 against the scoped rows only
#   3. LLM ranker     EVERY candidate-bearing pair is judged via LLM Mesh -
#                     pick one of the listed OIDs or refuse. String scores
#                     generate candidates; they never certify them.
# Unmapped pairs are safe by design: production writes LLM-generated rules
# from the names alone. Measured on the ground-truth book: 94% of what it
# maps is correct; coverage is bounded by library breadth, not this logic.
# Needs rapidfuzz. CAVEAT: _norm strips every non-ASCII character, so on
# non-Latin documents (or a non-Latin library) mapping coverage will be ~0%.
if RUN_OID_MAPPING:
    import csv
    import re

    import pandas as pd

    sys.modules.pop('oid_mapping', None)  # same rerun hygiene as the import cell
    import oid_mapping
    assert os.path.abspath(oid_mapping.__file__).startswith(
        os.path.abspath(MODULES_DIR) + os.sep), 'oid_mapping imported from outside the bundle'

    def _norm(s):
        s = re.sub(r'\(.*?\)', ' ', str(s or '').lower())
        s = re.sub(r'[^a-z0-9 ]+', ' ', s)
        return re.sub(r'\s+', ' ', s).strip()

    _lib_df = dataiku.Dataset(ECS_INDEX_DATASET).get_dataframe()
    _need = {'form_field_value', 'variable_name'}
    assert _need <= set(_lib_df.columns), ECS_INDEX_DATASET + ' must have columns ' + str(_need)
    _lib_df = _lib_df.fillna('')  # NaN is truthy - without this it becomes the string 'nan'
    _lib = []
    for _, _r in _lib_df.iterrows():
        _fv = str(_r.get('form_field_value') or '').strip()
        _vn = str(_r.get('variable_name') or '').strip()
        _fnorm = _norm(_fv)
        if not (_fv and _vn and _fnorm):  # drop rows whose label normalizes away
            continue
        _lib.append({'field': _fnorm, 'form': _norm(_r.get('form_name', '')),
                     'oid': _vn, 'field_raw': _fv,
                     'form_raw': str(_r.get('form_name', '')).strip()})
    print('library entries:', len(_lib))
    _formless = sum(1 for _e in _lib if not _e['form'])
    if _formless > len(_lib) // 2:
        # without form names layer-1 scoping matches nothing -> universal
        # unmapped that LOOKS like safe abstention but is a dataset problem
        print('WARNING:', _formless, 'of', len(_lib), 'library rows have no '
              'form_name - form scoping will unmap nearly everything')

    _done = _skipped = 0
    for _key in sorted(os.listdir(OUT_DIR)):
        _src = os.path.join(OUT_DIR, _key, 'fields_codegen_' + tag + '.csv')
        if not os.path.isdir(os.path.join(OUT_DIR, _key)):
            continue
        if not os.path.isfile(_src):
            _skipped += 1
            continue
        with open(_src, encoding='utf-8') as f:
            _pairs = sorted({(_norm(r['form_name']), _norm(r['field_name']))
                             for r in csv.DictReader(f)})
        try:
            # one Mesh failure must not sink the batch (rank_cases commits
            # nothing on failure, so there is no half-ranked state to persist)
            _results = oid_mapping.map_pairs(_pairs, _lib, llm=call_mesh)
        except Exception as _e:
            print(f'{_key[:52]:52s} mapping FAILED: {_e!r}')
            continue
        _rows = [{'form_name': r.form, 'field_name': r.field, 'status': r.status,
                  'oid': r.oid or '', 'via': r.via if r.status == 'mapped' else '',
                  'n_candidates': len(r.candidates)}
                 for r in _results]
        pd.DataFrame(_rows).to_csv(os.path.join(OUT_DIR, _key, 'oid_mapping_' + tag + '.csv'),
                                   index=False)
        _done += 1
        _mapped = [r for r in _results if r.status == 'mapped']
        _by = {v: sum(1 for r in _mapped if r.via == v)
               for v in sorted({r.via for r in _mapped})}
        print(f'{_key[:52]:52s} pairs={len(_pairs):5d} mapped={len(_mapped):4d} '
              f'({100 * len(_mapped) // max(1, len(_pairs))}%) by={_by}')
    print('mapped', _done, 'document(s);', _skipped,
          'dir(s) had no fields_codegen_' + tag + '.csv (check tag/run)')
    print('re-run the upload cell to persist the mapping CSVs')
else:
    print('RUN_OID_MAPPING is False - skipped')


## Reading the outputs

Per document (in the output folder, under `<doc_key>/`):

| artifact | meaning |
|---|---|
| `clusters.json` | stage-0 layout clusters, representative pages, selected `theta`, `status` |
| `rep_p<N>.txt` | the representative page dumps the LLM saw (text + geometry) |
| `codegen_prompt.txt` | the exact induction prompt |
| `codegen_reply_<tag>_<n>.py` | every parser version the LLM wrote |
| `codegen_reply_<tag>_confirm.txt` | the coverage-confirmation reply, if that round ran |
| `codegen_trail_<tag>.json` | per-version metrics/problems/audit + `stop_reason` + `best_version` |
| `generated_extractor_<tag>.py` | the accepted (best) parser |
| `fields_codegen_<tag>.csv` | final `form_name, field_name, page` extraction |
| `oid_mapping_<tag>.csv` | optional name->OID funnel result: status / oid / via (mapping cell) |

Summary `status` values: `ok` (clean audit, no warnings) / `ok_with_warnings`
(soft quality signals; document may legitimately violate them) /
`ok_audit_issues` (best version still has known page-level issues - review) /
`ok_unaudited` (audit errored; parser passed gates but was never page-verified) /
`needs_manual_template` (every version hard-failed - human review) /
`export_failed` (best parser accepted but the final replay crashed - re-run) /
`error` (unexpected per-document failure; see the cell output for the traceback) /
`skipped_encrypted`, `skipped_no_text_layer`, `skipped_no_pages`,
`skipped_missing_stage0` (stage 0 refused or failed on the PDF; OCR is out of
scope by design - fail loudly, never guess). A row may additionally carry
`text_layer_pct` when the book is PARTIALLY scanned (>=20% text pages proceed,
but the scanned pages are unreachable and their fields cannot appear in the
output - review such documents).

**Not in this notebook**: ground-truth evaluation (no annotated truth exists on
the Dataiku side; the local repo has `eval_form_field.py` for the one document
with printed OIDs) and the production OID-assignment step (form-scoped
candidates + LLM ranking - designed separately; the mapping cell here is the
lexical baseline only).
